# Hatteras Island CASCADE Hindcast
Step through each section, run it, look at the plot, confirm it's right before moving on.
Single configuration only (one `START_YEAR`, one `Hs`) — the sensitivity sweep stays in `HAT_groin_sensitivity_sweep.py`.

Pseudocode only below — move your own working code in from `HAT_hindcast_1984_2024_groinTest.py` section by section. Line numbers referenced are from that file.

## 1. Imports

In [ ]:
import base64
import os
import sys
from pathlib import Path

# cascade_pipeline and hatteras_site_config live in scripts/, which isn't installed
_here = Path.cwd().resolve()
_repo_root = next((p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()), None)
if _repo_root is None:
    raise RuntimeError(f"CASCADE repo root not found above {_here}")
SCRIPTS_DIR = _repo_root / "scripts"
# HAT_hindcast_config lives in scripts/hatteras_ms/ beside the runner. The .py
# gets that directory on sys.path for free; the notebook has to add it, and
# both files must reach the module the same way.
HATTERAS_MS_DIR = SCRIPTS_DIR / "hatteras_ms"
for _path in (SCRIPTS_DIR, HATTERAS_MS_DIR):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

# --- the settings file, read before anything it selects ----------------------
# `hat_run.yaml` in scripts/hatteras_ms/ is where a run is chosen: edit it,
# save it, Run All. See HAT_hindcast_config for the precedence rules.
#
# It is read HERE rather than in section 3 because two of its values are
# settled at import time and cannot be changed afterwards:
#
#   use_sandbox_cascade  decides which Cascade class cascade_pipeline.hindcast
#                        binds, which happens the moment it is imported below.
#                        It is pinned True and has no line in the yaml -- see
#                        the "not settable here" block at the foot of that
#                        file for why it must not follow groin.enabled.
#   show_figures         decides the matplotlib backend, which is fixed by the
#                        first figure created.
#
# Everything else is re-read in section 3, so editing the yaml and re-running
# that cell applies the change without restarting the kernel. Section 3 also
# checks these two against the file as it stands then, and raises if they have
# changed -- a stale sandbox flag is the failure where the groin silently does
# nothing.
from HAT_hindcast_config import load_run_config

_BOOT_CONFIG = load_run_config()
os.environ["CASCADE_USE_SANDBOX"] = (
    "1" if _BOOT_CONFIG.use_sandbox_cascade else "0")

# `output.show_figures: null` means "whichever file is being run decides".
# Here that is True: a notebook draws its figures inline. The headless .py
# reads the same null as False. The files written to disk are identical.
SHOW_FIGURES = (True if _BOOT_CONFIG.show_figures is None
                else _BOOT_CONFIG.show_figures)

import matplotlib
if not SHOW_FIGURES:
    matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess


from cascade_pipeline import reports
from cascade_pipeline.annotations import AnnotationConfig
from cascade_pipeline.hindcast import (
    DAM_TO_M,
    USE_SANDBOX_CASCADE,
    brie_r_ipl,
    build_background_erosion,
    build_cascade,
    build_domain_file_paths,
    build_shoreline_target,
    build_target_table,
    groin_trapping_schedule,
    load_barrier3d_contract,
    build_island_offset,
    island_offset_tilts,
    load_island_offset_dam,
    load_storm_series,
    measure_groin_extent,
    run_cascade_simulation,
    scenario_run_name,
)
from cascade_pipeline.plotting import setup_qc
from cascade_pipeline.coastsat_loess import (
    DEFAULT_LOESS,
    CoastSatDataset,
    LoessConfig,
    build_coastsat_series,
)
from cascade_pipeline.plotting.rate_comparison import (
    plot_annotated_rate_comparison,
    plot_rate_comparison,
)
from cascade_pipeline.plotting.shoreline_gif import (
    GifConfig,
    make_all_shoreline_gifs,
    make_shoreline_gif,
)
from cascade_pipeline.run_info import RunInfo
from cascade_pipeline.run_registry import (
    append_run_index,
    git_provenance,
    guard_run_dir,
    run_dir_contents,
    skill_vs_target,
    values_digest,
    timestamp,
    RUN_INDEX_FILENAME,
    write_run_metadata,
)
from cascade_pipeline.shoreline import (build_shoreline_matrix,
                                        compute_change_rate, compute_lrr)
from IPython.display import HTML, display

from hatteras_site_config import (
    HATTERAS_ANNOTATIONS,
    HATTERAS_BE_PRESETS,
    HATTERAS_BE_EDGE_DOMAINS,
    HATTERAS_BE_RATES_2004_IS_PLACEHOLDER,
    HATTERAS_DOMAINS,
    HATTERAS_PERIODS,
    resolve_be_preset,
)

print(f"Imports OK from {SCRIPTS_DIR}")
print(f"SHOW_FIGURES = {SHOW_FIGURES}")
print(f"USE_SANDBOX_CASCADE = {USE_SANDBOX_CASCADE}")
print(f"HATTERAS_DOMAINS.total_domains = {HATTERAS_DOMAINS.total_domains}")

## 2. Fixed dune/topo (period-independent) — QC plot
Same 2009 init surface for both periods, doesn't depend on `START_YEAR`.

Broken into steps so each can be confirmed before moving on:

| Step | What it does | What to check |
| --- | --- | --- |
| 2.1 | Resolve project paths | printed dirs all say `ok` |
| 2.2 | Build the 120 padded file paths | buffer/real/buffer order, 120 each |
| 2.3 | Verify every file exists | no exception |
| 2.4 | Units check vs Barrier3D contract | every check reports 90/90 |
| 2.5 | Browse per-domain extractor figures | dune picks and water trim, domain by domain |

### 2.1 Project paths

Everything derives from `SCRIPTS_DIR` (Section 1), so no absolute paths are
hardcoded. Layout under `data/hatteras_init/1-barrier3d-domains/`:

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#21-project-paths)

In [ ]:
# There is no TOPO_DUNE_INIT_YEAR any more. The arrays carry no year - the
# period is the PRODUCT DIRECTORY - and this notebook no longer builds their
# names; build_domain_file_paths() delegates to hat_topo_version.

# WHICH EXTRACTION -- resolved, not pinned. topo_dirs() reads VERSION out of
# HAT_dune_topo_extractor.py, so the runner, the dune-start road setbacks and
# the audits always describe the same extraction, and a version missing from
# disk is a loud error rather than a silently stale read.
#
# Pinning it by hand is what let this runner sit on 2009_v3 while the setbacks
# were rebuilt on v4 (2026-08-19) and then v5 (2026-08-20). The setback is
# measured from interior row 0, so that combination places the road against a
# row that does not exist on the grid being run.
#
# To reproduce an older run deliberately:
#     topo_dirs("2004-start", override="v3").
from hat_topo_version import topo_dirs  # scripts/, on sys.path from section 1
from hat_topo_version import BUFFER_DIR as _BUFFER_DIR

# _BOOT_CONFIG, not RUN_CONFIG: the period must be known HERE, and RUN_CONFIG is
# not loaded until section 3. The two are compared a few lines below section 3's
# reload, and section 3.0 re-asserts that this product matches the period that
# actually ran, so a boot/run divergence cannot silently pick the wrong barrier.
TOPO_PRODUCT = HATTERAS_PERIODS[_BOOT_CONFIG.start_year]["topo_product"]

_TOPO_DIR, _DUNE_DIR, TOPO_DUNE_VERSION = topo_dirs(TOPO_PRODUCT)
print(f"topography            {TOPO_PRODUCT} / {TOPO_DUNE_VERSION}  "
      f"(product from the period, version resolved)")

HATTERAS_DATA_BASE = PROJECT_BASE_DIR / "data" / "hatteras_init"
OUTPUT_ROOT = PROJECT_BASE_DIR / "output" / "raw_runs"
COASTSAT_BASE_DIR = (PROJECT_BASE_DIR / "scripts" / "input_prep"
                     / "5-scr" / "CoastSat")
PARAMETER_FILE = "Hatteras-CASCADE-parameters.yaml"  # resolved by CASCADE

BARRIER3D_DIR = HATTERAS_DATA_BASE / "1-barrier3d-domains"
# Taken from what topo_dirs() RETURNED rather than re-joined from parts. The
# old line rebuilt the path independently, which is how a resolver gets bypassed
# without anyone noticing - the same failure mode as HAT_road_elevation.py.
DUNE_TOPO_DIR = _TOPO_DIR.parent
BUFFER_DIR = _BUFFER_DIR

os.chdir(PROJECT_BASE_DIR)
# Runs are filed per period: section 3 builds OUTPUT_BASE_DIR =
# OUTPUT_ROOT / "<start>_<end>" once START_YEAR is expanded, so a
# 1984-2004 run and a 2004-2024 run of one scenario cannot land beside
# each other. run_index.csv stays at the root, covering both periods.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

reports.path_inventory([
    ("PROJECT_BASE_DIR", PROJECT_BASE_DIR),
    ("HATTERAS_DATA_BASE", HATTERAS_DATA_BASE),
    ("DUNE_TOPO_DIR", DUNE_TOPO_DIR),
    ("BUFFER_DIR", BUFFER_DIR),
    ("COASTSAT_BASE_DIR", COASTSAT_BASE_DIR),
    ("OUTPUT_ROOT", OUTPUT_ROOT),
])

### 2.2 Build the padded file lists

CASCADE takes one elevation file and one dune file per padded domain, in
alongshore order: 15 buffers, then real GIS 1–90, then 15 buffers (120 total).
Both buffer ends reuse the same sample profile, so only the real domains vary.

The returned lists are index-aligned with the padded array, so
`HATTERAS_DOMAINS.gis_to_pad(gis_id)` indexes straight into them.

In [ ]:



ELEVATION_FILE_PATHS, DUNE_FILE_PATHS = build_domain_file_paths(
    HATTERAS_DOMAINS, TOPO_PRODUCT)

_first_real = HATTERAS_DOMAINS.start_real_index
_last_real = HATTERAS_DOMAINS.end_real_index - 1

print(f"{len(ELEVATION_FILE_PATHS)} elevation + {len(DUNE_FILE_PATHS)} dune "
      f"paths (expect {HATTERAS_DOMAINS.total_domains} each)\n")
print("Padded array boundaries:")
for _label, _pad in [
    ("pad 0 (buffer)", 0),
    (f"pad {_first_real} (GIS {HATTERAS_DOMAINS.first_gis_id})", _first_real),
    (f"pad {_last_real} (GIS {HATTERAS_DOMAINS.last_gis_id})", _last_real),
    (f"pad {HATTERAS_DOMAINS.total_domains - 1} (buffer)",
     HATTERAS_DOMAINS.total_domains - 1),
]:
    print(f"  {_label:<22} {Path(ELEVATION_FILE_PATHS[_pad]).name}")

### 2.3 Verify every file exists

Checks all 240 paths up front. A stale `TOPO_DUNE_VERSION` or a moved data
folder fails here with a count and the first offender, rather than surfacing
as an empty plot or an opaque traceback inside Barrier3D's init.

In [ ]:
_expected_files = 2 * HATTERAS_DOMAINS.total_domains
_missing = [path for path in ELEVATION_FILE_PATHS + DUNE_FILE_PATHS
            if not Path(path).exists()]

if _missing:
    raise FileNotFoundError(
        f"{len(_missing)} of {_expected_files} init files missing. Check "
        f"TOPO_DUNE_VERSION ({TOPO_DUNE_VERSION!r}) and DUNE_TOPO_DIR.\n"
        f"  First missing: {_missing[0]}")

print(f"All {_expected_files} init files present.")

### 2.4 Units check against Barrier3D's input contract

Barrier3D's `load_input.py` converts the *scalar* YAML parameters but loads the
elevation and dune `.npy` files **verbatim** — `InteriorDomain` is whatever the
file contains, with no unit conversion applied:

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#24-units-check-against-barrier3ds-input-contract)

In [ ]:






BARRIER3D_CONTRACT = load_barrier3d_contract(HATTERAS_DATA_BASE / PARAMETER_FILE)

print(f"Contract from {PARAMETER_FILE}:")
print(f"  BarrierLength -> {BARRIER3D_CONTRACT['barrier_length_cells']} "
      f"alongshore cells")
print(f"  MHW           -> {BARRIER3D_CONTRACT['mhw_dam']:.3f} dam")
print(f"  BermEl        -> {BARRIER3D_CONTRACT['berm_el_dam']:.3f} dam "
      f"above MHW\n")

reports.run_units_check(ELEVATION_FILE_PATHS, DUNE_FILE_PATHS,
                        BARRIER3D_CONTRACT, HATTERAS_DOMAINS)

## 3. Island orientation — set `START_YEAR`

`START_YEAR` is the one flip in this notebook. It selects a period from
`HATTERAS_PERIODS` (in `hatteras_site_config.py`), and everything in Section 4
follows from it: run length, RSLR rate, storm series, background-erosion
preset, plus the road-setback and nourishment settings used later.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#3-island-orientation-set-startyear)

In [ ]:
from cascade_pipeline import nourishment
from hatteras_site_config import HATTERAS_NOURISHMENT_PROJECTS

# The run-selecting values in this section come from `hat_run.yaml`, read
# through HAT_hindcast_config: edit that file, save it, re-run this cell. A
# driver can also set them through the environment without editing anything,
# and interactively you can still type over any assignment below -- the loaded
# value is only the default, and the assignment is still the last word.
#
# load_run_config() RE-READS the yaml on every call rather than reusing the
# object section 1 built. The kernel caches the module, so a fresh read is the
# only way an edit to the yaml applies without a restart.

from HAT_hindcast_config import (
    load_run_config, describe as _describe_run_config, preflight as _preflight)

RUN_CONFIG = load_run_config()

# The two values section 1 already spent. They select an import and a
# matplotlib backend, so a yaml edited after section 1 ran would be reported
# in the block below while the kernel is still running the old choice -- and
# for the sandbox flag that means a groin-on run whose groin silently does
# nothing. Raised, not warned.
_BOOT_DRIFT = {
    name: (getattr(_BOOT_CONFIG, name), getattr(RUN_CONFIG, name))
    for name in ("use_sandbox_cascade", "show_figures")
    if getattr(_BOOT_CONFIG, name) != getattr(RUN_CONFIG, name)
}
if _BOOT_DRIFT:
    raise RuntimeError(
        "hat_run.yaml changed after section 1 imported on the old values:\n"
        + "\n".join(f"  {name}: section 1 used {was!r}, the file now says "
                     f"{now!r}" for name, (was, now) in _BOOT_DRIFT.items())
        + "\nThese are settled at import. Restart the kernel and Run All.")

START_YEAR = RUN_CONFIG.start_year   # 1984 or 2004

# The source/sink axis of the run matrix. Each name states a hypothesis about
# where the alongshore sediment budget is unresolved:
#   "zeroBE"   nothing imposed anywhere
#   "edgeBE"   only the two end domains, absorbing the open-boundary artifact
#   "calibBE"  the full per-domain fit against the CoastSat LRR
SOURCE_SINK_PRESET = RUN_CONFIG.source_sink_preset

# =============================================================================
# SCENARIO -- the management combination this run simulates
# =============================================================================
# The management axis of the run matrix, named rather than typed as four
# booleans, so a scenario is chosen in one word and cannot be assembled wrong
# by accident. What each switch controls:
#
#   roadway      roadway_manager: bulldozing, dune rebuild to the design
#                elevation, setback tracking, road drowning. Off leaves NC-12
#                as forcing that nothing acts on -- still loaded, still
#                audited in 5.1, never handed to a RoadwayManager.
#   beach_dune   beach_dune_manager, module and all: overwash filtering, the
#                fixed dune line (dune_migration_on = False), the 50 m
#                community-width drowning check, and fills. Off is the ONLY
#                way to get natural shoreline behaviour in the village
#                domains -- see the section 6 markdown on what is always-on.
#   fills        Within an enabled beach_dune_manager, whether the historical
#                fill is actually spent. False leaves the module and its
#                footprint exactly as they are, so a fills-on / fills-off pair
#                differs in the fill and nothing else.
#   relocations  Historical NC-12 relocation events. False in every named
#                scenario; read 5.1 before overriding it on.
SCENARIOS = {
    # nothing human acts on the island: the counterfactual
    "natural": dict(roadway=False, beach_dune=False,
                    fills=False, relocations=False),
    # NC-12 defended, villages left to behave naturally
    "roadway_only": dict(roadway=True, beach_dune=False,
                         fills=False, relocations=False),
    # villages managed and nourished, the road passive
    "beachdune_only": dict(roadway=False, beach_dune=True,
                           fills=True, relocations=False),
    # everything: the status-quo hindcast
    "full_management": dict(roadway=True, beach_dune=True,
                            fills=True, relocations=False),
    # everything except the sand -- isolates the fills against full_management
    "full_no_fill": dict(roadway=True, beach_dune=True,
                         fills=False, relocations=False),
}

SCENARIO = RUN_CONFIG.scenario

# Read here rather than beside the offset load in section 4: the run-name
# preview below needs it, and a value the name depends on must be settled
# before the name is predicted.
OFFSET_MODE = RUN_CONFIG.offset_mode

# The groin is NOT part of the scenario, deliberately. 12.3 measures its effect
# against a paired no-groin baseline identical in every other token, so every
# scenario is run twice -- False first to create the baseline, then True.
# Folding it into the table would double the table to say the same thing.
GROIN_ENABLED = RUN_CONFIG.groin_enabled

print("\n" + _describe_run_config())

# --- expand ------------------------------------------------------------------
if SCENARIO not in SCENARIOS:
    raise ValueError(f"SCENARIO must be one of {sorted(SCENARIOS)}, "
                     f"got {SCENARIO!r}")
_SCENARIO_PRESET = SCENARIOS[SCENARIO]
ENABLE_ROADWAY_MANAGEMENT = _SCENARIO_PRESET["roadway"]
ENABLE_BEACH_DUNE_MANAGEMENT = _SCENARIO_PRESET["beach_dune"]
ENABLE_NOURISHMENT_FILLS = _SCENARIO_PRESET["fills"]
# HAT_RELOCATIONS overrides the scenario when set; None leaves the preset
# in charge. Either way the departure is detected just below and the
# `reloc` token still comes from the switch, not from SCENARIO.
ENABLE_HISTORICAL_ROAD_RELOCATIONS = (
    _SCENARIO_PRESET["relocations"] if RUN_CONFIG.relocations is None
    else RUN_CONFIG.relocations)

# --- one-off overrides -------------------------------------------------------
# Uncomment to depart from the named scenario for a single run. The departure
# is detected and printed below, and the run name is still derived from the
# switches rather than from SCENARIO, so an overridden run cannot be filed
# under the scenario label it departed from.
# ENABLE_ROADWAY_MANAGEMENT = False
# ENABLE_BEACH_DUNE_MANAGEMENT = False
# ENABLE_NOURISHMENT_FILLS = False
# ENABLE_HISTORICAL_ROAD_RELOCATIONS = True

_SCENARIO_DEPARTURES = {
    key: (want, got) for key, want, got in (
        ("roadway", _SCENARIO_PRESET["roadway"], ENABLE_ROADWAY_MANAGEMENT),
        ("beach_dune", _SCENARIO_PRESET["beach_dune"],
         ENABLE_BEACH_DUNE_MANAGEMENT),
        ("fills", _SCENARIO_PRESET["fills"], ENABLE_NOURISHMENT_FILLS),
        ("relocations", _SCENARIO_PRESET["relocations"],
         ENABLE_HISTORICAL_ROAD_RELOCATIONS),
    ) if want != got
}

DAM_TO_M = 10.0   # Barrier3D works in decameters

if START_YEAR not in HATTERAS_PERIODS:
    raise ValueError(f"START_YEAR must be one of {sorted(HATTERAS_PERIODS)}, "
                     f"got {START_YEAR}")

# Normalised to the canonical key. The deprecated aliases ("base",
# "calibrated") still run, but it is the canonical name that reaches
# RUN_NAME in 7.5 -- an alias can never put a stale token in a directory name.
SOURCE_SINK_PRESET, _PRESET_BY_PERIOD = resolve_be_preset(SOURCE_SINK_PRESET)

PERIOD = HATTERAS_PERIODS[START_YEAR]

# Section 2 picked the topography from _BOOT_CONFIG.start_year, before
# RUN_CONFIG existed. If those disagree the run would model the wrong barrier
# entirely, so it is checked rather than assumed.
if PERIOD["topo_product"] != TOPO_PRODUCT:
    raise SystemExit(
        f"\n[stop] topography product mismatch.\n"
        f"  section 2 loaded : {TOPO_PRODUCT}  (from boot start_year "
        f"{_BOOT_CONFIG.start_year})\n"
        f"  this period wants: {PERIOD['topo_product']}  (START_YEAR "
        f"{START_YEAR})\n"
        f"The domain arrays already in memory are the wrong ones. Re-run "
        f"with a consistent start_year.\n")
END_YEAR = PERIOD["end_year"]
RUN_YEARS = END_YEAR - START_YEAR

SEA_LEVEL_RISE_RATE = PERIOD["sea_level_rise_rate"]
ENABLE_NOURISHMENT = PERIOD["enable_nourishment"]
NOURISHMENT_VOLUME = PERIOD["nourishment_volume"]
ISLAND_OFFSET_FILE = HATTERAS_DATA_BASE / PERIOD["island_offset_file"]
STORM_FILE = HATTERAS_DATA_BASE / PERIOD["storm_file"]
ROAD_SETBACK_FILE = HATTERAS_DATA_BASE / PERIOD["road_setback_file"]

# --- resolve the combinations that cannot both be true -----------------------
# A fill cannot land in a domain with no BeachDuneManager: 6.3 would report it
# as dropped and the run would nourish nothing. A relocation event cannot move
# a setback nothing reads. Both are resolved here and announced below, rather
# than left as a contradiction for a later cell to trip over.
_FILLS_FORCED_OFF = (ENABLE_NOURISHMENT_FILLS
                     and not ENABLE_BEACH_DUNE_MANAGEMENT)
if _FILLS_FORCED_OFF:
    ENABLE_NOURISHMENT_FILLS = False
_RELOCATIONS_FORCED_OFF = (ENABLE_HISTORICAL_ROAD_RELOCATIONS
                           and not ENABLE_ROADWAY_MANAGEMENT)
if _RELOCATIONS_FORCED_OFF:
    ENABLE_HISTORICAL_ROAD_RELOCATIONS = False

# Run name: the period stem only. The scenario suffix is derived
# from the active management switches in 7.5, once they all
# exist -- a hand-typed label is how a groin-off run ends up in a
# directory named for a groin-on one.
RUN_NAME_STEM = f"HAT_{START_YEAR}_{END_YEAR}"

# Run directories are filed by period. Resolved here, not in section 1,
# because the period is not known until START_YEAR is expanded above.
# Every later section derives its paths from OUTPUT_BASE_DIR -- RUN_DIR
# in 9, the paired groin baseline in 12.3 -- so scoping it here scopes
# all of them, and the baseline lookup can no longer resolve to a run
# from the other period.
PERIOD_TAG = f"{START_YEAR}_{END_YEAR}"
# Filed by period, then by source/sink preset. The preset directory is
# redundant with the preset token in RUN_NAME on purpose: the token is what
# run_index.csv, the logs and the figure captions all key on, and the
# directory is only there so the three presets of one period can be read
# side by side instead of interleaved in one listing of thirty-odd runs.
OUTPUT_BASE_DIR = OUTPUT_ROOT / PERIOD_TAG / SOURCE_SINK_PRESET
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"START_YEAR = {START_YEAR}  ->  {START_YEAR}-{END_YEAR}, "
      f"{RUN_YEARS} model years")
print(f"RUN_NAME_STEM = {RUN_NAME_STEM!r}"
      "   (scenario suffix derived in 7.5)")
print(f"SOURCE_SINK_PRESET = {SOURCE_SINK_PRESET!r}")
print(f"OUTPUT_BASE_DIR = {OUTPUT_BASE_DIR}")

# --- the name this scenario will produce, predicted from the switches --------
# Advisory only. 7.5 derives the authoritative RUN_NAME_BASE from what sections
# 5 and 6 actually built and raises if the two disagree, so this preview can
# never quietly become the thing that names the directory. Token order matches
# SCENARIO_SWITCHES in 7.5 exactly -- that is what makes the comparison valid.
# The period's fill is resolved with build_schedule, the same call 6 makes,
# rather than by re-implementing the date filter here.
_PERIOD_HAS_FILL = bool(nourishment.build_schedule(
    HATTERAS_NOURISHMENT_PROJECTS, HATTERAS_DOMAINS,
    START_YEAR, END_YEAR).projects)
_PREVIEW_TOKENS = [
    SOURCE_SINK_PRESET,
    None if OFFSET_MODE == "asrun" else f"offset{OFFSET_MODE}",
    "road" if ENABLE_ROADWAY_MANAGEMENT else "noroad",
    "reloc" if ENABLE_HISTORICAL_ROAD_RELOCATIONS else None,
    "bdm" if ENABLE_BEACH_DUNE_MANAGEMENT else "nobdm",
    ("nourish" if (ENABLE_NOURISHMENT_FILLS and _PERIOD_HAS_FILL)
     else ("nonourish" if _PERIOD_HAS_FILL and ENABLE_BEACH_DUNE_MANAGEMENT
           else None)),
    "groin" if GROIN_ENABLED else "nogroin",
]
RUN_NAME_PREVIEW = (f"{RUN_NAME_STEM}_"
                    + "_".join(t for t in _PREVIEW_TOKENS if t))

# What this run will be called, where it will land, whether something already
# lives there, and roughly how long it will take -- reported HERE rather than
# at the section 11 guard so a name collision is visible before sections 4-10
# do their work. Advisory: `guard_run_dir` in 11 is still the authority on
# what a collision does, and this does not duplicate that decision.
print("\n" + _preflight(RUN_NAME_PREVIEW,
                        OUTPUT_BASE_DIR / RUN_NAME_PREVIEW,
                        config=RUN_CONFIG))

reports.scenario_report(
    scenario=SCENARIO, scenarios=SCENARIOS, departures=_SCENARIO_DEPARTURES,
    roadway_on=ENABLE_ROADWAY_MANAGEMENT,
    relocations_on=ENABLE_HISTORICAL_ROAD_RELOCATIONS,
    relocations_forced_off=_RELOCATIONS_FORCED_OFF,
    beach_dune_on=ENABLE_BEACH_DUNE_MANAGEMENT,
    fills_on=ENABLE_NOURISHMENT_FILLS, fills_forced_off=_FILLS_FORCED_OFF,
    period_expects_nourishment=ENABLE_NOURISHMENT,
    groin_enabled=GROIN_ENABLED, run_name_preview=RUN_NAME_PREVIEW,
    input_files=[("island offset", ISLAND_OFFSET_FILE),
                 ("storms", STORM_FILE),
                 ("road setback", ROAD_SETBACK_FILE)])

### 3.1 Island orientation QC plot

Both periods are drawn so the active one is visible *in context* — the 1984
and 2004 lines should have the same overall shape (it is the same island) but
sit at different cross-shore positions. The lower panel is the difference
between the two files. It is **not** shoreline change:
`island_offset_hybrid.py:125` zeroes each year on its own most-seaward domain,
and that domain moved between the two surveys, so the difference carries a
constant offset (56.4 m for 1984-2004, enough to flip the sign of the mean).
Read it for the alongshore *pattern* only. Section 9.4 rebuilds the real change
from the raw transect files, which share a fixed datum.

Buffer domains are excluded here; their offsets are extrapolated ramps rather
than measured, so they would dominate the y-range. See the poster figure in
`scripts/figure_making/init_figure/` for the buffered plan view.

In [ ]:
# OFFSET_MODE selects which shoreline_offset variant is built; see
# cascade_pipeline.hindcast.build_island_offset. The default, "asrun",
# reproduces the historical unit error (the file is METRES and Cascade
# wants metres, but load_island_offset divided by 10), so previously
# published runs stay reproducible until the correction is adopted.
island_offset = build_island_offset(
    ISLAND_OFFSET_FILE, HATTERAS_DOMAINS, mode=OFFSET_MODE)
OFFSET_TILTS = island_offset_tilts(island_offset, HATTERAS_DOMAINS)
_offsets_by_year = {
    year: load_island_offset(
        HATTERAS_DATA_BASE / period["island_offset_file"], HATTERAS_DOMAINS)
    for year, period in HATTERAS_PERIODS.items()
}

_real = slice(HATTERAS_DOMAINS.start_real_index, HATTERAS_DOMAINS.end_real_index)
print(f"{START_YEAR} offsets: {island_offset.size} padded domains | "
      f"real span {island_offset[_real].min() * DAM_TO_M:.0f}-"
      f"{island_offset[_real].max() * DAM_TO_M:.0f} m")

# Setup QC figure. Nothing downstream reads it -- comment it out to skip.
_ = setup_qc.plot_island_orientation(_offsets_by_year, START_YEAR,
                                     HATTERAS_DOMAINS)
plt.show()

### 3.2 Initialization plan view

Section 2's topography and Section 3's offsets combined: every domain's
elevation array shifted cross-shore by its island offset, so the initial
island reads as a map. This is the last check before the forcings — if the
orientation year is wrong, or a domain's topography is misaligned, it shows
up here as a kink in the island rather than as a number.

Both variants are drawn. The buffered one shows the 15 interpolated domains
on each end; they repeat one sample profile stepped along the extrapolated
offset ramp, which is what makes the ramp itself visible.

Compositing — cross-shore padding, unit conversion, canvas assembly — comes
from `cascade_pipeline.plotting.init_planview`, which
`scripts/figure_making/init_figure/plot_initialization_poster_no_border.py`
also uses, so the two cannot disagree about the surface. Only the styling
differs: the poster is tuned for print, this is tuned for a notebook cell.

In [ ]:
from cascade_pipeline.plotting import init_planview

PLAN_VIEW = init_planview.PlanViewConfig()  # extractor defaults: 200 rows, -3.0 m water

# Setup QC figure. Nothing downstream reads it -- comment it out to skip.
# PLAN_VIEW itself is read again by 5.2, so it stays defined either way.
_ = setup_qc.plot_initialization_planview(
    ELEVATION_FILE_PATHS, island_offset, HATTERAS_DOMAINS, START_YEAR,
    config=PLAN_VIEW)
plt.show()

## 4. Period forcings — RSLR, storms, source/sink

Everything in this section is resolved by the `START_YEAR` set in Section 3.
Re-run Section 3 with the other year and re-run these cells: every number and
plot below should change together. That is the check — a forcing that does
*not* change is one that is not actually period-dependent.

### 4.1 Relative sea level rise

A single rate in m/yr, applied every model year. Both periods are plotted from
their own start year so the fork is visible: the 2004 period rises faster
(0.006 vs 0.004 m/yr) over the same 20-year run length.

In [ ]:
print(f"SEA_LEVEL_RISE_RATE = {SEA_LEVEL_RISE_RATE} m/yr")
print(f"  over {RUN_YEARS} years -> "
      f"{SEA_LEVEL_RISE_RATE * RUN_YEARS:.3f} m total rise")

# Setup QC figure. Nothing downstream reads it -- comment it out to skip.
_ = setup_qc.plot_sea_level_rise(HATTERAS_PERIODS, START_YEAR)
plt.show()

### 4.2 Storm series

Barrier3D's storm file is one row per storm with columns
`time, Rhigh, Rlow, period, duration`. `time` is the model time step the storm
lands on (1-based), `Rhigh`/`Rlow` are runup elevations in **decameters**, and
`duration` is in hours.

Two things to check: the time axis should span the run (roughly 1 to
`RUN_YEARS`, with no storms past the end), and `Rhigh` should look like metres
of runup once converted — a file left in metres would show up here as values
ten times too large.

In [ ]:



def plot_storm_series(series, start_year, run_years):
    """Plots storm runup and storm counts against the model time axis.

    Args:
        series: DataFrame from load_storm_series.
        start_year: Calendar year of model time step 0.
        run_years: Number of model years in the run.

    Returns:
        The matplotlib Figure.
    """
    fig, (ax_runup, ax_count) = plt.subplots(
        2, 1, figsize=(13, 7), sharex=True,
        gridspec_kw={"height_ratios": [2, 1]})

    ax_runup.scatter(series["time"], series["Rhigh_m"], s=18, alpha=0.7,
                     label="Rhigh")
    ax_runup.scatter(series["time"], series["Rlow_m"], s=18, alpha=0.7,
                     label="Rlow")
    ax_runup.set_ylabel("Runup elevation (m)")
    ax_runup.set_title(f"Storm series, {start_year}-{start_year + run_years}")
    ax_runup.legend()

    counts = series["time"].value_counts().sort_index()
    ax_count.bar(counts.index, counts.values, width=0.7)
    ax_count.set_ylabel("Storms per year")
    ax_count.set_xlabel("Model time step (year 1 = "
                        f"{start_year + 1})")

    for ax in (ax_runup, ax_count):
        ax.axvline(run_years + 0.5, color="#c0392b", ls="--", lw=1.2,
                   label="_run end")
        ax.grid(alpha=0.3)

    fig.tight_layout()
    return fig


STORM_SERIES = load_storm_series(STORM_FILE)

reports.storm_report(storms=STORM_SERIES, storm_file=STORM_FILE,
                     run_years=RUN_YEARS)

_ = plot_storm_series(STORM_SERIES, START_YEAR, RUN_YEARS)
plt.show()

### 4.3 Source/sink (background erosion)

Background erosion is CASCADE's stand-in for shoreline change driven by
alongshore transport gradients that the model does not resolve. It is a per
domain rate in m/yr, passed to Barrier3D as `Rat`. Sign convention, from
`cascade/brie_coupler.py`: **(-) = erosion, (+) = accretion**.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#43-sourcesink-background-erosion)

In [ ]:



def plot_background_erosion(presets, start_year, active_preset, geometry):
    """Plots every background-erosion preset for one period.

    Args:
        presets: Mapping of preset name to {start_year: {gis_id: rate}}.
        start_year: The period to plot.
        active_preset: Name of the preset currently selected.
        geometry: DomainGeometry used to slice out the real domains.

    Returns:
        The matplotlib Figure.
    """
    real = slice(geometry.start_real_index, geometry.end_real_index)
    gis_ids = np.arange(geometry.first_gis_id, geometry.last_gis_id + 1)

    fig, (ax_full, ax_zoom) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    interior = slice(1, -1)   # drop the locked end domains for the zoom range
    interior_rates = []

    for name, by_period in presets.items():
        rates = np.array(build_background_erosion(
            by_period[start_year], geometry))[real]
        interior_rates.append(rates[interior])
        is_active = name == active_preset
        style = dict(lw=2.0 if is_active else 1.2,
                     alpha=1.0 if is_active else 0.5,
                     marker="o" if is_active else None, ms=3,
                     label=f"{name}" + ("  (active)" if is_active else ""))
        ax_full.plot(gis_ids, rates, **style)
        ax_zoom.plot(gis_ids, rates, **style)

    # GIS 1 and 90 are locked to large solved values that would otherwise
    # flatten the interior detail, so the lower panel excludes them.
    margin = 0.5
    ax_zoom.set_ylim(min(r.min() for r in interior_rates) - margin,
                     max(r.max() for r in interior_rates) + margin)

    ax_full.set_title(f"Source/sink rates, {start_year} period "
                      "     (-) erosion  /  (+) accretion")
    ax_full.set_ylabel("Background erosion (m/yr)")
    ax_full.legend()

    ax_zoom.set_title(f"Zoomed to GIS {geometry.first_gis_id + 1}-"
                      f"{geometry.last_gis_id - 1} (locked end domains "
                      "off-scale)")
    ax_zoom.set_ylabel("Background erosion (m/yr)")
    ax_zoom.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")

    for ax in (ax_full, ax_zoom):
        ax.axhline(0, color="k", lw=0.8)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


DOMAIN_BE_RATES = HATTERAS_BE_PRESETS[SOURCE_SINK_PRESET][START_YEAR]
BACKGROUND_EROSION_RATES = build_background_erosion(
    DOMAIN_BE_RATES, HATTERAS_DOMAINS)
USE_BACKGROUND_EROSION = any(rate != 0.0 for rate in BACKGROUND_EROSION_RATES)

# A derived fact, not a switch: the preset decides it. Checked because the two
# used to be separate knobs that could disagree, and the run name carried both.
_EXPECT_BE_ON = SOURCE_SINK_PRESET != "zeroBE"
if USE_BACKGROUND_EROSION != _EXPECT_BE_ON:
    raise ValueError(
        f"preset {SOURCE_SINK_PRESET!r} implies "
        f"USE_BACKGROUND_EROSION={_EXPECT_BE_ON}, but the expanded rates give "
        f"{USE_BACKGROUND_EROSION}. The preset in hatteras_site_config.py does "
        f"not match its name.")

reports.background_erosion_report(
    preset=SOURCE_SINK_PRESET, start_year=START_YEAR,
    domain_rates=DOMAIN_BE_RATES, rates=BACKGROUND_EROSION_RATES,
    use_background_erosion=USE_BACKGROUND_EROSION, geometry=HATTERAS_DOMAINS,
    rates_2004_are_placeholder=HATTERAS_BE_RATES_2004_IS_PLACEHOLDER)

_ = plot_background_erosion(HATTERAS_BE_PRESETS, START_YEAR,
                            SOURCE_SINK_PRESET, HATTERAS_DOMAINS)
plt.show()

## 5. `roadway_manager` — setbacks, per-domain elevation, historical events

NC-12's forcing is three things: **where** the road sits (setback, by period),
**how high** it is (elevation, period-independent), and **which domains are
managed** at all.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#5-roadwaymanager-setbacks-per-domain-elevation-historical-events)

In [ ]:
from cascade_pipeline import roadway
from hatteras_site_config import (
    HATTERAS_COMMUNITY_ZONES,
    HATTERAS_FIRST_ROAD_DOMAIN,
    HATTERAS_LAST_ROAD_DOMAIN,
    HATTERAS_ROAD_ELEVATION_FILE,
    HATTERAS_ROAD_EVENTS,
)

# ENABLE_ROADWAY_MANAGEMENT and ENABLE_HISTORICAL_ROAD_RELOCATIONS are set in
# section 3, with the other management switches. The forcing below is loaded
# either way: with management off the setbacks and elevations are still read,
# audited in 5.1 and drawn in 5.2, they simply never reach a RoadwayManager.

ROADWAY = roadway.RoadwayConfig()
_road_span = (HATTERAS_FIRST_ROAD_DOMAIN, HATTERAS_LAST_ROAD_DOMAIN)

# --- setback: by period ------------------------------------------------------
# ROAD_SETBACK_FILE is resolved from PERIOD in section 3.
road_setbacks_full, _missing_setbacks = roadway.load_road_setbacks(
    ROAD_SETBACK_FILE, HATTERAS_DOMAINS, *_road_span)

# --- elevation: one set for every period -------------------------------------
ROAD_ELEVATION_FILE = HATTERAS_DATA_BASE / HATTERAS_ROAD_ELEVATION_FILE
road_elevation_full, _missing_elevations = roadway.load_road_elevations(
    ROAD_ELEVATION_FILE, HATTERAS_DOMAINS, *_road_span, config=ROADWAY)

# --- which domains CASCADE actually manages ----------------------------------
ROADWAY_MANAGEMENT_ON = roadway.build_roadway_management_on(
    HATTERAS_DOMAINS, *_road_span,
    community_zones=HATTERAS_COMMUNITY_ZONES,
    enabled=ENABLE_ROADWAY_MANAGEMENT)

_road_slice = slice(HATTERAS_DOMAINS.gis_to_pad(HATTERAS_FIRST_ROAD_DOMAIN),
                    HATTERAS_DOMAINS.gis_to_pad(HATTERAS_LAST_ROAD_DOMAIN) + 1)
reports.roadway_report(
    setback_file=ROAD_SETBACK_FILE, setbacks=road_setbacks_full,
    missing_setbacks=_missing_setbacks,
    elevation_file=ROAD_ELEVATION_FILE, elevations=road_elevation_full,
    missing_elevations=_missing_elevations, road_slice=_road_slice,
    config=ROADWAY, roadway_on=ENABLE_ROADWAY_MANAGEMENT,
    management_on=ROADWAY_MANAGEMENT_ON,
    first_road_gis=HATTERAS_FIRST_ROAD_DOMAIN,
    last_road_gis=HATTERAS_LAST_ROAD_DOMAIN,
    community_zones=HATTERAS_COMMUNITY_ZONES,
    road_events=HATTERAS_ROAD_EVENTS,
    relocations_enabled=ENABLE_HISTORICAL_ROAD_RELOCATIONS)


### 5.1 Pre-flight audit — which roads will not survive year one

`bulldoze` tests the two rows *flanking* the road — never the road's own cells
— and drowns it when either flank is more than 20 % water.

A drowned road is not a warning. `roadway_manager` sets `_drown_break` and
returns immediately on every later year, so the domain gets no overwash
removal, no dune rebuilding and no relocation for the rest of the run. It
becomes an **unmanaged barrier wearing a road label**, which is why these
domains are named here rather than discovered in the output: any
managed-versus-unmanaged comparison has to exclude them explicitly.


In [ ]:
road_audit = roadway.audit_setbacks(
    ELEVATION_FILE_PATHS, road_setbacks_full, HATTERAS_DOMAINS, *_road_span,
    management_on=ROADWAY_MANAGEMENT_ON, config=ROADWAY)
audit_summary = roadway.summarise_audit(road_audit)

reports.road_audit_report(audit=road_audit, summary=audit_summary)


### 5.2 The roadway on the initialization surface

The same plan view as §3.2, with NC-12 drawn where CASCADE puts it. Each
domain gets a horizontal bar at `offset + setback`, because that is literally
what the model does — `road_start = int(setback / cell)` is one row index
applied to every alongshore profile in the domain.

The orange bar is the position after the prescribed relocations, evaluated at
t=0. That is the **upper bound**: CASCADE decrements the setback by dune
migration between now and the event year, so the real relocated position sits
seaward of the bar drawn here. Blue markers flag the domains whose road drowns
in year one.


In [ ]:
from cascade_pipeline.plotting.road_planview import plot_roadway_planview

# Display only - nothing in this cell feeds the run.
road_setbacks_relocated = road_setbacks_full.copy()
for _event in HATTERAS_ROAD_EVENTS:
    if isinstance(_event, roadway.RelocationEvent) and _event.enabled:
        road_setbacks_relocated = roadway.relocated_setbacks(
            road_setbacks_relocated, _event, HATTERAS_DOMAINS)

road_offset_cells = np.round(
    island_offset * DAM_TO_M / PLAN_VIEW.cell_size_m).astype(int)

_fig = plot_roadway_planview(
    ELEVATION_FILE_PATHS,
    road_offset_cells,
    road_setbacks_full,
    HATTERAS_DOMAINS,
    title=(f"NC-12 on the CASCADE initialization  |  setbacks {START_YEAR}, "
           f"topography {TOPO_PRODUCT}/{TOPO_DUNE_VERSION}, dune offsets applied"),
    relocated_m=road_setbacks_relocated,
    drowning_gis=audit_summary["drowning"],
    config=PLAN_VIEW,
    xlabel="GIS domain (S -> N, Cape Point to Pea Island)",
)
plt.show()


## 6. `beach_dune_manager` — nourishment schedule + overwash filter

Two different things arrive bundled in one CASCADE module, and the distinction
matters more than the name suggests.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#6-beachdunemanager-nourishment-schedule-overwash-filter)

### 6.1 Build the schedule, filter, and management footprints
Everything CASCADE is handed for this period: the period-filtered project
schedule, the overwash filter, and the two per-domain module footprints.
`NOURISHMENT_VOLUME_INIT` is deliberately zero -- see the comment in the cell.


In [ ]:
from cascade_pipeline import nourishment
from hatteras_site_config import (
    HATTERAS_BEACH_DUNE,
    HATTERAS_NOURISHMENT_PROJECTS,
)

# --- schedule: one project list, period-filtered ------------------------------
# Every Hatteras project falls in 2004-2024, so a 1984 run builds an empty
# schedule from this same list rather than needing a period-keyed one.
BN_SCHEDULE = nourishment.build_schedule(
    HATTERAS_NOURISHMENT_PROJECTS, HATTERAS_DOMAINS, START_YEAR, END_YEAR)

# --- the schedule the model is actually driven by -----------------------------
# ENABLE_NOURISHMENT_FILLS (section 3). Suppressing fills builds an EMPTY
# schedule for the same period rather than skipping apply_to_cascade: the loop
# still rewrites nourish_now and nourishment_volume to zero every year, and 6.3
# and 12.2 both check against what was driven rather than what was intended.
# BN_SCHEDULE itself is left intact and still defines the module footprint
# below, so turning fills off changes the fill and not the footprint.
BN_SCHEDULE_APPLIED = BN_SCHEDULE if ENABLE_NOURISHMENT_FILLS else (
    nourishment.build_schedule([], HATTERAS_DOMAINS, START_YEAR, END_YEAR))

# --- what CASCADE is handed ---------------------------------------------------
# The filter is inert with the module off -- only BeachDuneManager applies it --
# so it is built as zeros there rather than left at its community values. The
# array handed to CASCADE should state what the run does, not what it would do.
OVERWASH_FILTER = (
    nourishment.build_overwash_filter(
        HATTERAS_DOMAINS, HATTERAS_COMMUNITY_ZONES, config=HATTERAS_BEACH_DUNE)
    if ENABLE_BEACH_DUNE_MANAGEMENT
    else [0.0] * HATTERAS_DOMAINS.total_domains)
OVERWASH_TO_DUNE = HATTERAS_BEACH_DUNE.overwash_to_dune_pct
BEACH_DUNE_MANAGEMENT_ON = nourishment.build_beach_dune_management_on(
    HATTERAS_DOMAINS, HATTERAS_COMMUNITY_ZONES, BN_SCHEDULE.nourished_gis,
    enabled=ENABLE_BEACH_DUNE_MANAGEMENT)

# Placeholder for the Cascade() call. Every value is rewritten each model year
# by BN_SCHEDULE.apply_to_cascade(), so this is 0 rather than
# PERIOD["nourishment_volume"]: if the schedule ever fails to reach the model,
# a year should nourish nothing rather than quietly nourish the default.
NOURISHMENT_VOLUME_INIT = [0.0] * HATTERAS_DOMAINS.total_domains

DOUBLE_MANAGED_GIS = nourishment.find_double_managed(
    BEACH_DUNE_MANAGEMENT_ON, ROADWAY_MANAGEMENT_ON, HATTERAS_DOMAINS)
BN_AUDIT = nourishment.audit_schedule(
    BN_SCHEDULE_APPLIED, BEACH_DUNE_MANAGEMENT_ON, config=HATTERAS_BEACH_DUNE)

### 6.2 Plot helpers
`contiguous_gis_runs` is generic domain geometry; `plot_beach_dune_management`
draws the two panels below. Both are candidates to move into
`cascade_pipeline` -- nothing in this cell produces output.


In [ ]:
def contiguous_gis_runs(mask, geometry):
    """Contiguous GIS spans where a padded boolean mask is True.

    Args:
        mask: Sequence of booleans indexed by padded position.
        geometry: DomainGeometry describing the padded array.

    Returns:
        A list of inclusive (first_gis, last_gis) tuples.
    """
    runs = []
    start = None
    for pad in range(geometry.start_real_index, geometry.end_real_index):
        if mask[pad] and start is None:
            start = pad
        elif not mask[pad] and start is not None:
            runs.append((int(geometry.pad_to_gis(start)),
                         int(geometry.pad_to_gis(pad - 1))))
            start = None
    if start is not None:
        runs.append((int(geometry.pad_to_gis(start)),
                     int(geometry.pad_to_gis(geometry.end_real_index - 1))))
    return runs


def plot_beach_dune_management(overwash_filter, beach_dune_on, roadway_on,
                               schedule, double_managed_gis, geometry):
    """Plots the overwash filter and both management footprints by GIS domain.

    Two panels on a shared domain axis. The top panel is the magnitude the
    module actually applies; the bottom is which modules run where, so the
    overlap that neither footprint was drawn to avoid is visible rather than
    inferred.

    Args:
        overwash_filter: Per-domain filter percentages.
        beach_dune_on: Per-domain booleans for beach_nourishment_module.
        roadway_on: Per-domain booleans for roadway_management_module.
        schedule: The NourishmentSchedule for this period.
        double_managed_gis: GIS domains where both modules run.
        geometry: DomainGeometry describing the padded array.

    Returns:
        The matplotlib Figure.
    """
    c_filter = "#08519C"    # deep blue, matches the notebook's obs family
    c_bdm = "#08519C"
    c_road = "#6BAED6"      # lighter step of the same hue
    c_nourish = "#FF8C00"   # warm accent, as elsewhere for model-side forcing
    c_double = "#B71C1C"

    real = slice(geometry.start_real_index, geometry.end_real_index)
    gis_ids = np.arange(geometry.first_gis_id, geometry.last_gis_id + 1)
    filter_real = np.asarray(overwash_filter, dtype=float)[real]

    nourished_pads = {geometry.gis_to_pad(g) for g in schedule.nourished_gis}
    nourished_mask = [pad in nourished_pads
                      for pad in range(geometry.total_domains)]

    fig, (ax_filter, ax_map) = plt.subplots(
        2, 1, figsize=(13, 6), sharex=True,
        gridspec_kw=dict(height_ratios=[1.4, 1]))

    # --- top: how much overwash is filtered, and where -----------------------
    # The shading is the point of the panel: nourished ground and filtered
    # ground only partly coincide, and the domains that are shaded but have no
    # bar are the ones running the module for the fill alone.
    for first_gis, last_gis in contiguous_gis_runs(nourished_mask, geometry):
        ax_filter.axvspan(first_gis - 0.5, last_gis + 0.5, color=c_nourish,
                          alpha=0.15, zorder=0, label="_nolegend_")
    ax_filter.bar(gis_ids, filter_real, width=0.9, color=c_filter,
                  label="overwash filtered (developed ground)")
    if schedule.nourished_gis:
        ax_filter.axvspan(np.nan, np.nan, color=c_nourish, alpha=0.15,
                          label="nourished at least once")
    ax_filter.set_ylabel("overwash_filter (%)")
    ax_filter.set_ylim(0, max(50, filter_real.max() * 1.25))
    ax_filter.set_title(
        f"beach_dune_manager forcing, {schedule.start_year}-"
        f"{schedule.end_year}     "
        f"filter {HATTERAS_BEACH_DUNE.community_overwash_filter_pct:.0f}% + "
        f"{HATTERAS_BEACH_DUNE.overwash_to_dune_pct:.0f}% to dune")
    ax_filter.legend(loc="upper left", frameon=False)
    ax_filter.grid(alpha=0.3, axis="y")

    # --- bottom: which module runs where -------------------------------------
    rows = [
        ("roadway_manager", roadway_on, c_road, 0),
        ("beach_dune_manager", beach_dune_on, c_bdm, 1),
    ]
    for label, mask, color, y in rows:
        for first_gis, last_gis in contiguous_gis_runs(mask, geometry):
            ax_map.barh(y, last_gis - first_gis + 1, left=first_gis - 0.5,
                        height=0.55, color=color, align="center")
        ax_map.text(geometry.first_gis_id - 1.5, y, label, ha="right",
                    va="center", fontsize=9)

    # Overlap: outlined rather than filled, so the underlying rows stay legible.
    for first_gis, last_gis in contiguous_gis_runs(
            [beach_dune_on[p] and roadway_on[p]
             for p in range(geometry.total_domains)], geometry):
        ax_map.add_patch(plt.Rectangle(
            (first_gis - 0.5, -0.45), last_gis - first_gis + 1, 1.9,
            fill=False, edgecolor=c_double, lw=1.8, hatch="///", zorder=5))

    # Nourishment events, labelled per project rather than per year -- two
    # projects share 2022, and a single averaged label would sit between them,
    # over neither.
    for row in schedule.events():
        ax_map.plot(row["gis"], 2, marker="v", ms=8, color=c_nourish, zorder=6)
    for project in schedule.projects:
        ax_map.text(float(np.mean(project.gis_domains)), 2.4,
                    str(project.year), ha="center", va="bottom", fontsize=9,
                    color=c_nourish)
    ax_map.text(geometry.first_gis_id - 1.5, 2, "nourishment", ha="right",
                va="center", fontsize=9, color=c_nourish)

    if double_managed_gis:
        ax_map.legend(
            handles=[plt.Rectangle((0, 0), 1, 1, fill=False,
                                   edgecolor=c_double, lw=1.8, hatch="///")],
            labels=[f"both modules ({len(double_managed_gis)} domains)"],
            loc="upper left", frameon=False, fontsize=9)

    ax_map.set_ylim(-0.7, 3.4)
    ax_map.set_yticks([])
    ax_map.set_xlim(geometry.first_gis_id - 6, geometry.last_gis_id + 1)
    ax_map.set_xticks(np.arange(geometry.first_gis_id,
                                geometry.last_gis_id + 1, 5))
    ax_map.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")
    ax_map.grid(alpha=0.3, axis="x")
    for spine in ("top", "right", "left"):
        ax_map.spines[spine].set_visible(False)

    fig.tight_layout()
    return fig

### 6.3 Audit and QC plot
The schedule as it will be applied, the pre-flight audit, and the
double-management warning. The plot shows the overwash filter over the
footprints, so the overlap neither footprint was drawn to avoid is visible
rather than inferred.


In [ ]:
reports.beach_dune_report(
    start_year=START_YEAR, end_year=END_YEAR,
    beach_dune_enabled=ENABLE_BEACH_DUNE_MANAGEMENT,
    fills_enabled=ENABLE_NOURISHMENT_FILLS,
    roadway_enabled=ENABLE_ROADWAY_MANAGEMENT,
    config=HATTERAS_BEACH_DUNE, management_on=BEACH_DUNE_MANAGEMENT_ON,
    overwash_to_dune=OVERWASH_TO_DUNE,
    community_zones=HATTERAS_COMMUNITY_ZONES,
    schedule=BN_SCHEDULE, schedule_applied=BN_SCHEDULE_APPLIED,
    audit=BN_AUDIT, double_managed=DOUBLE_MANAGED_GIS,
    geometry=HATTERAS_DOMAINS)

_ = plot_beach_dune_management(
    OVERWASH_FILTER, BEACH_DUNE_MANAGEMENT_ON, ROADWAY_MANAGEMENT_ON,
    BN_SCHEDULE_APPLIED, DOUBLE_MANAGED_GIS, HATTERAS_DOMAINS)
plt.show()

## 7. `hard_structures` / groin -- Buxton groin field

`cascade/groin.py` attaches to a run through `cascade._groin_callback` and is
called once per model year from inside `Cascade.update()`, immediately before
the alongshore-transport solve (`cascade/cascade_groin.py:600`). Each active
year it adds `-M` to the updrift domain and `+M` to the downdrift domain of
`x_s_dt`, then hands the array on. BRIE's implicit diffusion solve spreads that
dipole in the same step, so the fillet's taper and alongshore extent are
**emergent** -- only its amplitude is imposed.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#7-hardstructures-groin----buxton-groin-field)

### 7.1 Switches, structure, and the sediment-budget reference
Every constant the groin needs, and nothing that acts on them. The
`GROIN_ENABLED` guard fails fast here because attaching a callback to a
Cascade built from the real package is a silent no-op.


In [ ]:
from cascade.groin import GroinCallback, predict_fillet

# =============================================================================
# Switches
# =============================================================================
# GROIN_ENABLED is set in section 3, beside the scenario, so one cell decides
# what a run simulates. The guard below stays here: this is where a wrongly
# configured groin would silently do nothing.

# The pre-AST hook exists ONLY in cascade/cascade_groin.py. Attaching a callback
# to a Cascade built from the real package is a silent no-op: the run succeeds,
# the groin does nothing, and the output looks like a valid groin-on run.
if GROIN_ENABLED and not USE_SANDBOX_CASCADE:
    raise RuntimeError(
        "GROIN_ENABLED=True requires USE_SANDBOX_CASCADE=True (section 1). "
        "cascade.cascade.Cascade.update() has no _groin_callback hook, so the "
        "groin would silently do nothing.\n"
        "USE_SANDBOX_CASCADE is pinned True and has no line in hat_run.yaml, "
        "so reaching this means HAT_USE_SANDBOX_CASCADE=0 is set in the "
        "environment. Unset it, or turn the groin off.")

# =============================================================================
# Structure: the Buxton groin field
# =============================================================================
# Net alongshore transport on Hatteras is southward, so updrift = north. The
# two domains must be adjacent -- they share the blocked boundary at GIS 5.5.
GROIN_UPDRIFT_GIS = 6       # source: accretes
GROIN_DOWNDRIFT_GIS = 5     # sink:   erodes
GROIN_INSTALL_YEAR = 1969   # confirmed construction date

# --- amplitude: the only tunable knob ----------------------------------------
# Both knobs come from HAT_hindcast_config so the sweep driver can set them
# per run. They are fit JOINTLY across the two periods, not independently: over
# 1984-2004 the cumulative trapping is M*(16 + 4f), so f barely separates from
# M there, while over 2004-2024 the run sits entirely past the 2003 ramp and
# only the product M*f is identifiable. Neither window pins both on its own.
GROIN_TRAPPING_RATE_M_YR = RUN_CONFIG.groin_trapping_rate_m_yr
GROIN_M_PROVENANCE = ("joint two-period fit against the CoastSat D6-D5 "
                      "differential; see output/groin_sweep/ for the M-f "
                      "ridge and which grid bounds the solution touches")

# --- deterioration: 1996 last repair -> 2003 storm damage --------------------
GROIN_DETERIORATION_DELAY_YEARS = 1996 - GROIN_INSTALL_YEAR   # = 27
GROIN_DETERIORATION_MODE = "linear_ramp"
GROIN_DETERIORATION_RAMP_YEARS = 2003 - 1996                  # = 7
GROIN_DETERIORATION_FRACTION = RUN_CONFIG.groin_deterioration_fraction

# =============================================================================
# Sediment-budget reference
# =============================================================================
REACH_TRANSPORT_LOSS_M3_YR = 5.9e5
REACH_TRANSPORT_CITATION = ("Inman & Dolan (1989), via Moore et al. (2010), "
                            "doi:10.1029/2009JF001299")
REACH_TRANSPORT_CAVEAT = (
    "reach-integrated transport-gradient LOSS, Oregon Inlet to Cape Hatteras "
    "(~60 km) -- a divergence, not a gross flux at Buxton, so this bounds "
    "order of magnitude only")

# Active profile height, converting shoreline displacement to volume. The yaml
# carries DShoreface=22.25 with LShoreface=2640 (a 1:119 slope, consistent with
# meters), but Barrier3D's internal length unit is decameters and no conversion
# is applied at load (barrier3d.py:1181). Both candidates print here rather
# than one being asserted; section 11 resolves it from the constructed model.
GROIN_PROFILE_HEIGHT_CANDIDATES_M = (12.0, 24.0)

GROIN_PLOT_HALFWIDTH = 5   # domains either side of the groin in the QC plot

### 7.2 Build the callback
`GROIN_CB` is built unconditionally so 7.4's QC output renders either way.
`GROIN_CALLBACK` is the one attached to the model in section 11, and is `None`
when the groin is off.


In [ ]:
# =============================================================================
# Build
# =============================================================================
# Built unconditionally so the QC output below renders either way. Only
# GROIN_CALLBACK is attached to the model, in section 11.
GROIN_CB = GroinCallback(
    updrift_pad=HATTERAS_DOMAINS.gis_to_pad(GROIN_UPDRIFT_GIS),
    downdrift_pad=HATTERAS_DOMAINS.gis_to_pad(GROIN_DOWNDRIFT_GIS),
    trapping_rate_m_yr=GROIN_TRAPPING_RATE_M_YR,
    start_year=START_YEAR,
    install_year=GROIN_INSTALL_YEAR,
    n_domains=HATTERAS_DOMAINS.total_domains,
    deterioration_delay_years=GROIN_DETERIORATION_DELAY_YEARS,
    deterioration_mode=GROIN_DETERIORATION_MODE,
    deterioration_ramp_years=GROIN_DETERIORATION_RAMP_YEARS,
    deterioration_fraction=GROIN_DETERIORATION_FRACTION,
)
GROIN_CALLBACK = GROIN_CB if GROIN_ENABLED else None

### 7.3 Diagnostic and plot helpers
Two diagnostics and one plot. `groin_trapping_schedule` reads the
deterioration curve without stepping the callback, so querying it here leaves
the run's year counter at zero.


In [ ]:


def plot_groin_setting(offset_dam, callback, geometry, updrift_gis,
                       downdrift_gis, halfwidth, periods, active_year):
    """Plots the groin's alongshore setting, its forcing, and its schedule.

    Three panels. The top two share a GIS axis spanning `halfwidth` domains
    either side of the structure: the initial shoreline the groin is imposed
    on, and the dipole imposed on it. The bottom panel is the deterioration
    schedule for every period, so the ramp's position within the active run is
    visible rather than inferred.

    Arrow direction is deliberately not drawn on the top panel: the sign
    convention of the island-offset file is not the sign convention of
    `x_s_dt`, and conflating them on one axis is how a fillet ends up drawn on
    the wrong side. The forcing panel carries the signed values instead.

    Args:
        offset_dam: Padded island-offset array, in decameters.
        callback: The GroinCallback being configured.
        geometry: DomainGeometry describing the padded array.
        updrift_gis: GIS domain that accretes.
        downdrift_gis: GIS domain that erodes.
        halfwidth: Domains either side of the groin to show.
        periods: Mapping of start year to a period dict carrying "end_year".
        active_year: The period currently selected.

    Returns:
        The matplotlib Figure.
    """
    c_shore = "#08519C"     # obs family, as elsewhere in the notebook
    c_groin = "#B71C1C"     # ANN_C_GROIN
    c_accrete = "#FF8C00"   # model-side forcing
    c_erode = "#6BAED6"

    boundary = (downdrift_gis + updrift_gis) / 2.0
    gis_lo = max(geometry.first_gis_id, downdrift_gis - halfwidth)
    gis_hi = min(geometry.last_gis_id, updrift_gis + halfwidth)
    gis_ids = np.arange(gis_lo, gis_hi + 1)
    pads = geometry.gis_to_pad(gis_ids)
    offset_m = np.asarray(offset_dam)[pads] * DAM_TO_M

    fig, (ax_shore, ax_force, ax_time) = plt.subplots(
        3, 1, figsize=(13, 9),
        gridspec_kw=dict(height_ratios=[1.5, 1, 1.2]))

    # --- top: the shoreline the groin is imposed on --------------------------
    ax_shore.plot(gis_ids, offset_m, color=c_shore, lw=2.0, marker="o", ms=4,
                  label=f"{active_year} island offset")
    ax_shore.axvline(boundary, color=c_groin, lw=1.6, ls=":",
                     label=f"Buxton groin (GIS {boundary})")
    for gis, color, label in (
            (updrift_gis, c_accrete, f"D{updrift_gis} updrift (accretes)"),
            (downdrift_gis, c_erode, f"D{downdrift_gis} downdrift (erodes)")):
        ax_shore.axvspan(gis - 0.5, gis + 0.5, color=color, alpha=0.18,
                         zorder=0, label=label)
    ax_shore.set_ylabel("Island offset (m)")
    ax_shore.set_title(
        f"Buxton groin setting, GIS {gis_lo}-{gis_hi}     "
        f"updrift D{updrift_gis} (north) / downdrift D{downdrift_gis} (south)"
        f"     M = {callback.M:.0f} m/yr")
    ax_shore.legend(loc="best", frameon=False, fontsize=9)

    # --- middle: the forcing, in x_s_dt's own sign convention ----------------
    forcing = np.zeros_like(gis_ids, dtype=float)
    forcing[gis_ids == updrift_gis] = -callback.M
    forcing[gis_ids == downdrift_gis] = +callback.M
    colors = [c_accrete if v < 0 else c_erode if v > 0 else "#DDDDDD"
              for v in forcing]
    ax_force.bar(gis_ids, forcing, width=0.9, color=colors)
    ax_force.axvline(boundary, color=c_groin, lw=1.6, ls=":")
    ax_force.axhline(0, color="k", lw=0.8)
    ax_force.set_ylim(-callback.M * 1.5, callback.M * 1.5)
    ax_force.set_ylabel("x_s_dt applied (m/yr)")
    ax_force.set_title("Injected dipole, full strength")
    ax_force.text(0.01, 0.94,
                  "(-) seaward advance = accretion\n"
                  "(+) landward retreat = erosion",
                  transform=ax_force.transAxes, ha="left", va="top",
                  fontsize=8, color="#444444")
    # Labels go INSIDE the bars. The 1.5x headroom leaves room for them there,
    # and text beyond the bar tips collides with the title and the x label.
    for _gis, _value, _text in (
            (updrift_gis, -callback.M, f"-M\nD{updrift_gis} accretes"),
            (downdrift_gis, +callback.M, f"+M\nD{downdrift_gis} erodes")):
        ax_force.text(_gis, _value * 0.5, _text, ha="center", va="center",
                      fontsize=8, color="white", fontweight="bold")

    for ax in (ax_shore, ax_force):
        ax.set_xlim(gis_lo - 0.6, gis_hi + 0.6)
        ax.set_xticks(gis_ids)
        ax.grid(alpha=0.3, axis="y")
    ax_force.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")

    # --- bottom: deterioration schedule, every period ------------------------
    # Built from throwaway callbacks so the configured one is never stepped.
    for year in sorted(periods):
        probe = GroinCallback(
            updrift_pad=callback.updrift_pad,
            downdrift_pad=callback.downdrift_pad,
            trapping_rate_m_yr=callback.M,
            start_year=year,
            install_year=callback.install_year,
            n_domains=callback.n_domains,
            deterioration_delay_years=callback.deterioration_delay_years,
            deterioration_mode=callback.deterioration_mode,
            deterioration_ramp_years=callback.deterioration_ramp_years,
            deterioration_fraction=callback.deterioration_fraction,
        )
        years, m_eff = groin_trapping_schedule(
            probe, year, periods[year]["end_year"])
        is_active = year == active_year
        ax_time.plot(years, m_eff, lw=2.2 if is_active else 1.2,
                     alpha=1.0 if is_active else 0.45,
                     marker="o" if is_active else None, ms=3,
                     label=f"{year}-{periods[year]['end_year']}"
                           + ("  (active)" if is_active else ""))

    if callback.deterioration_year is not None:
        ax_time.axvline(callback.deterioration_year, color=c_groin, lw=1.2,
                        ls="--", alpha=0.8)
        ax_time.axvline(
            callback.deterioration_year + callback.deterioration_ramp_years,
            color=c_groin, lw=1.2, ls="--", alpha=0.8)
        ax_time.axvspan(
            callback.deterioration_year,
            callback.deterioration_year + callback.deterioration_ramp_years,
            color=c_groin, alpha=0.08,
            label=f"ramp {callback.deterioration_year:.0f}-"
                  f"{callback.deterioration_year + callback.deterioration_ramp_years:.0f}")

    ax_time.set_ylim(0, callback.M * 1.15)
    ax_time.set_ylabel("M applied (m/yr)")
    ax_time.set_xlabel("Model year")
    ax_time.set_title("Deterioration schedule, both periods     "
                      "1996 last repair -> 2003 storm damage")
    ax_time.legend(loc="best", frameon=False, fontsize=9)
    ax_time.grid(alpha=0.3)

    fig.tight_layout()
    return fig

### 7.4 Report -- configuration, sediment budget, double-count audit
The configured groin, the implied sediment transfer against the reach budget,
and the overlap with the calibrated source/sink rates. Both tensions are
reported, not corrected; section 11 resolves the profile height from the
constructed model and section 12 reports the resulting misfit.


In [ ]:
# =============================================================================
# Report
# =============================================================================
reports.groin_report(
    enabled=GROIN_ENABLED, callback=GROIN_CB,
    updrift_gis=GROIN_UPDRIFT_GIS, downdrift_gis=GROIN_DOWNDRIFT_GIS,
    install_year=GROIN_INSTALL_YEAR, start_year=START_YEAR, end_year=END_YEAR,
    geometry=HATTERAS_DOMAINS,
    trapping_rate_m_yr=GROIN_TRAPPING_RATE_M_YR,
    m_provenance=GROIN_M_PROVENANCE,
    deterioration_mode=GROIN_DETERIORATION_MODE,
    deterioration_delay_years=GROIN_DETERIORATION_DELAY_YEARS,
    deterioration_ramp_years=GROIN_DETERIORATION_RAMP_YEARS,
    deterioration_fraction=GROIN_DETERIORATION_FRACTION,
    profile_height_candidates_m=GROIN_PROFILE_HEIGHT_CANDIDATES_M,
    reach_transport_loss_m3_yr=REACH_TRANSPORT_LOSS_M3_YR,
    reach_transport_citation=REACH_TRANSPORT_CITATION,
    reach_transport_caveat=REACH_TRANSPORT_CAVEAT,
    source_sink_preset=SOURCE_SINK_PRESET, domain_be_rates=DOMAIN_BE_RATES)

_ = plot_groin_setting(
    island_offset, GROIN_CB, HATTERAS_DOMAINS,
    GROIN_UPDRIFT_GIS, GROIN_DOWNDRIFT_GIS, GROIN_PLOT_HALFWIDTH,
    HATTERAS_PERIODS, START_YEAR)
plt.show()

### 7.5 Scenario summary -- every switch, and the run name derived from them

This notebook runs one scenario at a time, and the switches that decide which
one are spread across sections 3, 5, 6 and 7. Nothing below changes the model.
It exists because `RUN_NAME_SUFFIX` used to be typed by hand: run with the
groin off, forget to retype the label, and the output lands in a directory
named for a different experiment. The suffix is now derived from the switches
themselves, so the directory name cannot disagree with what was simulated.

One switch deliberately contributes no token: background erosion. It is
implied by the source/sink preset and checked against it in 4.3, so emitting
both produced names like `..._base_noBE_...` that said the same thing twice
without saying which zero it was.

Output directories from earlier runs (`HAT_1984_2004_full_calibrated`) will not
match the derived names, and neither will runs made before the presets were
renamed. That is the trade: the old names are shorter, the new ones are true.

In [ ]:
# Every switch that changes what is simulated, and the run name derived from
# them. Ordered as the sections are: period, forcing, then management.
SCENARIO_SWITCHES = [
    ("period", f"{START_YEAR}-{END_YEAR} ({RUN_YEARS} yr)", None),
    ("source/sink preset", SOURCE_SINK_PRESET, SOURCE_SINK_PRESET),
    # No token when "asrun": every run predating the shoreline_offset
    # unit finding used it, and adding a token would rename them all.
    ("shoreline offset", OFFSET_MODE,
     None if OFFSET_MODE == "asrun" else f"offset{OFFSET_MODE}"),
    # No token of its own: it is implied by the preset, and checked against it
    # in 4.3. Emitting both produced names like "..._base_noBE_..." that said
    # the same thing twice without saying which zero it was.
    ("background erosion", USE_BACKGROUND_EROSION, None),
    ("roadway management", f"{sum(ROADWAY_MANAGEMENT_ON)} domains"
     if ENABLE_ROADWAY_MANAGEMENT else "off",
     "road" if ENABLE_ROADWAY_MANAGEMENT else "noroad"),
    ("historical relocations", ENABLE_HISTORICAL_ROAD_RELOCATIONS,
     "reloc" if ENABLE_HISTORICAL_ROAD_RELOCATIONS else None),
    ("beach/dune manager", f"{sum(BEACH_DUNE_MANAGEMENT_ON)} domains"
     if ENABLE_BEACH_DUNE_MANAGEMENT else "off",
     "bdm" if ENABLE_BEACH_DUNE_MANAGEMENT else "nobdm"),
    # "nonourish" only when there was fill to withhold and a module to
    # withhold it in: with beach_dune_manager off, "nobdm" already says no
    # fill, and 1984 has no project to suppress. The project count is
    # deliberately not in the name -- it is a property of the period, not of
    # the scenario, and run_index.csv carries it as nourishment_projects.
    ("nourishment fills", f"{len(BN_SCHEDULE_APPLIED.projects)} applied"
     if ENABLE_NOURISHMENT_FILLS
     else ("suppressed" if BN_SCHEDULE.projects else "none in period"),
     "nourish" if BN_SCHEDULE_APPLIED.projects
     else ("nonourish" if BN_SCHEDULE.projects and ENABLE_BEACH_DUNE_MANAGEMENT
           else None)),
    ("groin", "on" if GROIN_ENABLED else "off",
     "groin" if GROIN_ENABLED else "nogroin"),
]

RUN_NAME_SUFFIX = "_".join(
    token for _, _, token in SCENARIO_SWITCHES if token)
RUN_NAME_BASE = f"{RUN_NAME_STEM}_{RUN_NAME_SUFFIX}"

# The section 3 preview was predicted from the switches; this name is derived
# from what sections 5 and 6 actually built. A difference means a switch did
# not reach the module it names -- the failure that produces a run filed under
# a scenario it did not simulate. Raised, not warned.
if RUN_NAME_BASE != RUN_NAME_PREVIEW:
    raise AssertionError(
        f"run name disagrees with the section 3 preview\n"
        f"  section 3    {RUN_NAME_PREVIEW}\n"
        f"  section 7.5  {RUN_NAME_BASE}\n"
        f"The preview follows the switches; this follows the built modules.")

reports.scenario_summary_report(
    scenario=SCENARIO, departures=_SCENARIO_DEPARTURES,
    switches=SCENARIO_SWITCHES, run_name_base=RUN_NAME_BASE,
    double_managed=DOUBLE_MANAGED_GIS, groin_enabled=GROIN_ENABLED,
    updrift_gis=GROIN_UPDRIFT_GIS,
    beach_dune_on_updrift=BEACH_DUNE_MANAGEMENT_ON[
        HATTERAS_DOMAINS.gis_to_pad(GROIN_UPDRIFT_GIS)])

## 8. CoastSat target rates -- LOESS windows

This section builds the observational target. The `calibBE` source/sink
preset in section 4.3 was fit against the curve produced here, and section 12
draws the model against it -- so what this section decides is what the model is
judged by. It emits `cs_series` for the section 12 figures and
`COASTSAT_TARGET`, a per-domain table of the target rate itself.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#8-coastsat-target-rates----loess-windows)

### 8.1 Load both periods' CoastSat series
Observational data, not simulation forcing, so both periods always load and
the active one is selected from them. A `START_YEAR` with no matching dataset
fails here rather than silently plotting the wrong period.


In [ ]:
from cascade_pipeline.annotations import add_geographic_annotations
from cascade_pipeline.coastsat_loess import compute_domain_means
from cascade_pipeline.plotting.rate_comparison import DEFAULT_RATE_COMPARISON

# =============================================================================
# Datasets
# =============================================================================
# Both periods always load. This is observational data, not simulation forcing,
# and section 12 needs the non-active period available for reference styling --
# so it is deliberately NOT folded into HATTERAS_PERIODS.
COASTSAT_DATASETS = [
    CoastSatDataset(
        label="CoastSat LRR (1984-2004)",
        period_start=1984,
        csv_path=str(COASTSAT_BASE_DIR / "1984_2004" / "transect_lrr_full.csv"),
    ),
    CoastSatDataset(
        label="CoastSat LRR (2004-2024)",
        period_start=2004,
        csv_path=str(COASTSAT_BASE_DIR / "2004_2024" / "transect_lrr_full.csv"),
    ),
]

LOESS_CONFIG = LoessConfig(window_domains=(7, 10), skip_southern_domains=10)

# Named here rather than left implicit. rate_comparison resolves the reference
# window as max(window_domains); this makes that choice visible, and the
# assertion below catches a window list whose maximum is not what was intended.
TARGET_WINDOW = 10
if TARGET_WINDOW != max(LOESS_CONFIG.window_domains):
    raise ValueError(
        f"TARGET_WINDOW={TARGET_WINDOW} but rate_comparison will use "
        f"max(window_domains)={max(LOESS_CONFIG.window_domains)} as the "
        f"reference curve -- section 12 would compare against a different "
        f"curve than the one reported here.")

cs_series = build_coastsat_series(
    COASTSAT_DATASETS, active_period_start=START_YEAR,
    loess_config=LOESS_CONFIG, domains=HATTERAS_DOMAINS)

CS_ACTIVE = next((cs for cs in cs_series if cs["active"]), None)
if CS_ACTIVE is None:
    raise RuntimeError(
        f"no CoastSat dataset has period_start == START_YEAR ({START_YEAR}); "
        f"loaded: {[cs['period_start'] for cs in cs_series]}")

### 8.2 The target, as a table
`build_target_table` resolves each domain to a LOESS window or a raw rate;
`groin_differential` extracts the updrift-minus-downdrift signal that is the
observational check on the dipole section 7 imposes.


In [ ]:
# =============================================================================
# The target, as a table
# =============================================================================



COASTSAT_TARGET = build_target_table(
    CS_ACTIVE, LOESS_CONFIG, HATTERAS_DOMAINS, TARGET_WINDOW)

### 8.3 Plot helper
`plot_coastsat_target` draws the target and its provenance. Defined
separately so it can be edited and re-run without rebuilding the table.


In [ ]:
# =============================================================================
# Plot
# =============================================================================
def plot_coastsat_target(cs, target, loess_config, geometry, annotations,
                         config, window, updrift_gis, downdrift_gis,
                         zoom_to_gis=15):
    """Plots the active period's transect scatter and its smoothed target.

    Two panels. The top is the whole island with the geographic annotation
    layer, so the target can be read against the villages and shoal zones that
    explain its structure. The bottom zooms on the unsmoothed southern zone,
    where the target is per-domain means rather than a curve and where the
    groin sits.

    Colors come from RateComparisonConfig so this figure and the section 12
    model comparison stay in the same visual language.

    Args:
        cs: The active entry from build_coastsat_series.
        target: Output of build_target_table.
        loess_config: LoessConfig used.
        geometry: DomainGeometry.
        annotations: AnnotationConfig for the geographic layer.
        config: RateComparisonConfig supplying colors and marker sizes.
        window: Reference window width, in domains.
        updrift_gis, downdrift_gis: The groin's flanking domains.
        zoom_to_gis: Last GIS domain shown in the lower panel.

    Returns:
        The matplotlib Figure.
    """
    skip = loess_config.skip_southern_domains
    # Matches rate_comparison's GIS-domain x-axis exactly, so the scatter in
    # this figure lands where it lands in the section 12 comparison.
    scatter_x = (np.asarray(cs["transect_along_coast"])
                 / geometry.domain_spacing_m + geometry.first_gis_id)
    boundary = (updrift_gis + downdrift_gis) / 2.0

    fig, (ax_full, ax_zoom) = plt.subplots(
        2, 1, figsize=(14, 9),
        gridspec_kw=dict(height_ratios=[1.7, 1]))

    # --- top: the whole island ----------------------------------------------
    ax_full.scatter(scatter_x, cs["transect_rates"], color=config.raw_color,
                    s=config.raw_scatter_size, alpha=config.raw_scatter_alpha,
                    zorder=1, linewidths=0, label="transect LRR")
    for win in cs["windows"]:
        color = config.window_colors.get(win["window"],
                                         config.window_color_default)
        mask = win["gis_x"] > skip
        is_target = win["window"] == window
        ax_full.plot(win["gis_x"][mask], win["smoothed"][mask], color=color,
                     lw=2.2 if is_target else 1.6, zorder=4 if is_target else 3,
                     label=f"LOESS {win['window']}-dom"
                           + ("  (target)" if is_target else ""))
    ax_full.axvspan(geometry.first_gis_id - 0.5, skip + 0.5, color="#AAAAAA",
                    alpha=0.16, zorder=0,
                    label=f"D1-{skip}: no LOESS (raw means)")
    add_geographic_annotations(ax_full, annotations)
    ax_full.axhline(0, color="k", lw=0.8)
    ax_full.set_ylabel("Shoreline change rate (m/yr)")
    ax_full.set_title(f"{cs['label']}     target = LOESS {window}-domain "
                      f"({window * geometry.domain_spacing_m / 1000:.1f} km) "
                      f"north of D{skip}, raw per-domain mean south of it")
    ax_full.set_xlim(geometry.first_gis_id - 1, geometry.last_gis_id + 1)
    ax_full.legend(loc="upper left", frameon=True, fontsize=8, ncol=2)
    ax_full.grid(alpha=0.3)

    # --- bottom: the unsmoothed zone, where the groin is ---------------------
    south = target[target["gis_domain"] <= zoom_to_gis]
    in_zoom = scatter_x <= zoom_to_gis + 1
    ax_zoom.scatter(scatter_x[in_zoom],
                    np.asarray(cs["transect_rates"])[in_zoom],
                    color=config.raw_color, s=config.raw_scatter_size * 3,
                    alpha=0.75, zorder=2, linewidths=0, label="transect LRR")
    raw = south[south["source"].str.startswith("raw")]
    loess = south[~south["source"].str.startswith("raw")]
    ax_zoom.step(raw["gis_domain"], raw["target_lrr_m_yr"], where="mid",
                 color="#08519C", lw=2.0, zorder=4,
                 label=f"target: raw per-domain mean (D1-{skip})")
    if len(loess):
        ax_zoom.plot(loess["gis_domain"], loess["target_lrr_m_yr"],
                     color="#08519C", lw=2.0, ls="--", zorder=4,
                     label=f"target: LOESS {window}-dom (D{skip + 1}+)")
    for gis, color in ((updrift_gis, "#FF8C00"), (downdrift_gis, "#6BAED6")):
        ax_zoom.axvspan(gis - 0.5, gis + 0.5, color=color, alpha=0.20, zorder=0)
    ax_zoom.axvline(boundary, color="#B71C1C", lw=1.6, ls=":", zorder=5,
                    label=f"Buxton groin (GIS {boundary})")
    ax_zoom.axhline(0, color="k", lw=0.8)
    ax_zoom.set_xlim(geometry.first_gis_id - 0.6, zoom_to_gis + 0.6)
    ax_zoom.set_xticks(range(geometry.first_gis_id, zoom_to_gis + 1))
    ax_zoom.set_ylabel("Shoreline change rate (m/yr)")
    ax_zoom.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")
    ax_zoom.set_title(f"Unsmoothed southern zone, D{geometry.first_gis_id}-"
                      f"{zoom_to_gis}: the groin's target is a mean over a "
                      f"handful of transects")
    ax_zoom.legend(loc="best", frameon=True, fontsize=8)
    ax_zoom.grid(alpha=0.3)

    fig.tight_layout()
    return fig

### 8.4 Report and QC plot
How many domains came from a LOESS window versus a raw rate, the groin
differential, and the figure.


In [ ]:
# =============================================================================
# Report
# =============================================================================
reports.coastsat_report(
    target=COASTSAT_TARGET, active=CS_ACTIVE, target_window=TARGET_WINDOW,
    loess_config=LOESS_CONFIG, geometry=HATTERAS_DOMAINS, cs_series=cs_series,
    updrift_gis=GROIN_UPDRIFT_GIS, downdrift_gis=GROIN_DOWNDRIFT_GIS)

_ = plot_coastsat_target(
    CS_ACTIVE, COASTSAT_TARGET, LOESS_CONFIG, HATTERAS_DOMAINS,
    HATTERAS_ANNOTATIONS, DEFAULT_RATE_COMPARISON, TARGET_WINDOW,
    GROIN_UPDRIFT_GIS, GROIN_DOWNDRIFT_GIS)
plt.show()

## 9. Figure configuration

The plotting *functions* are already imported in section 1 -- `make_shoreline_gif`,
`make_all_shoreline_gifs`, `plot_rate_comparison`, `plot_annotated_rate_comparison`,
`build_shoreline_matrix`, `compute_change_rate`. This section is not those. It is
the configuration they need, gathered before section 12 calls them.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#9-figure-configuration)

### 9.1 Site config every figure needs
The keyword bundles section 12 splats into its plotting calls, plus the two
sign/extent conventions. Every entry point defaults to the package defaults
and `DEFAULT_ANNOTATIONS` is empty, so an omitted keyword loses the geographic
layer without raising -- hence the bundles.


In [ ]:
from cascade_pipeline.plotting.rate_comparison import DEFAULT_RATE_COMPARISON

# =============================================================================
# Site config every figure needs
# =============================================================================
# Splat these into the section 12 plotting calls. See the markdown above: every
# entry point defaults to the package defaults, and DEFAULT_ANNOTATIONS is empty,
# so an omitted keyword loses the geographic layer without raising.
RATE_FIG_KWARGS = dict(
    domains=HATTERAS_DOMAINS,
    annotations=HATTERAS_ANNOTATIONS,
    loess_config=LOESS_CONFIG,        # section 8's, not DEFAULT_LOESS
    config=DEFAULT_RATE_COMPARISON,
)

gif_config = GifConfig(
    fps=3,
    year_stride=1,
    annotate=True,
    auto_open=False,
    keep_frames=False,
    save_matrix=True,          # lets a later run difference against this one
    ocean_at_bottom=True,      # Hatteras' real cross-shore layout
    baseline_label="no-groin baseline",
    target_label=f"observed {END_YEAR} dune line",   # section 9.4's target
)

GIF_KWARGS = dict(
    domains=HATTERAS_DOMAINS,
    annotations=HATTERAS_ANNOTATIONS,
    gif_config=gif_config,
)

# Figure-level conventions the section 12 calls read.
FLIP_SIGN_MODEL = True          # x_s_TS increases landward; flip so up = seaward
PLOT_REAL_DOMAINS_ONLY = True   # GIS 1-90 axis; False adds the buffer domains

# WHICH ESTIMATOR THE FIGURES DRAW. "lrr" is the OLS slope through every
# annual state; "endpoint" is (x[-1] - x[0]) / RUN_YEARS. The observational
# target is an LRR -- CoastSat's transect_lrr_full.csv is a per-transect OLS
# slope with r_squared and unc_m_yr beside it -- so "lrr" is the only setting
# that puts the same quantity on both sides of the comparison. "endpoint"
# exists to redraw a pre-2026-08-22 figure, not as an alternative.
RATE_ESTIMATOR = "lrr"
if RATE_ESTIMATOR not in ("lrr", "endpoint"):
    raise ValueError(f"RATE_ESTIMATOR must be 'lrr' or 'endpoint', "
                     f"got {RATE_ESTIMATOR!r}")

# Below this, a domain's LRR is reported as a poor summary of its trajectory
# rather than passed over. 0.50 is a reporting threshold and nothing else --
# no value is dropped, filtered, or flagged in the CSV on account of it.
LRR_R2_FLOOR = 0.50

### 9.2 Animation jobs
One dict per GIF. `"groin"` fans out into one animation per structure in
`annotations.groins`, so a new structure is picked up without editing the list.


In [ ]:
# =============================================================================
# Animation jobs
# =============================================================================
# range: "real" | "all" | "groin" | "groin_span" | (gis_lo, gis_hi)
# mode:  "position" | "displacement" | "difference"
# pad:   half-width in domains, read only by "groin" / "groin_span"
#
# "groin" fans out into one GIF per structure in annotations.groins, so new
# structures are picked up without editing this list.
# `output.make_gifs: false` in hat_run.yaml empties this list rather than
# skipping the section 12 call. The shoreline matrix .npy is written by that
# same call, OUTSIDE the job loop, and section 12.3's paired groin baseline
# and HAT_scenario_grid.py both read it -- so short-circuiting the call would
# cost the run its matrix, while an empty job list costs only the animations.
GIF_JOBS = [
    dict(range="real", mode="displacement"),
    dict(range="real", mode="position"),
    dict(range="groin", mode="position", pad=9),
    dict(range="groin", mode="difference", pad=9),
] if RUN_CONFIG.make_gifs else []

### 9.3 Baseline for difference jobs
`scenario_run_name` rebuilds the run name from the same switch tokens section
7.5 used, so flipping `groin` cannot accidentally match inside `nogroin`. The
baseline `.npy` is located here if a no-groin run already exists.


In [ ]:
# =============================================================================
# Baseline for difference jobs
# =============================================================================



GIF_BASELINE_NAME = None
GIF_BASELINE_NPY = None
if GROIN_ENABLED:
    GIF_BASELINE_NAME = scenario_run_name(
        SCENARIO_SWITCHES, RUN_NAME_STEM, groin="nogroin")
    _baseline = (OUTPUT_BASE_DIR / GIF_BASELINE_NAME
                 / f"{GIF_BASELINE_NAME}_shoreline_matrix.npy")
    GIF_BASELINE_NPY = str(_baseline) if _baseline.exists() else None

### 9.4 Validation target -- where the island actually ended up

A `"position"` or `"displacement"` GIF already carries a dashed grey year-0
reference. This adds the other end of the comparison: a static line showing the
**surveyed** island position in the run's end year, so how close the run gets is
readable off the animation instead of only off the rate figure.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#94-validation-target----where-the-island-actually-ended-up)

In [ ]:
# =============================================================================
# Validation target -- the surveyed island position in the run's end year
# =============================================================================
# Read from the RAW transect CSVs, not the padded offset files: the padded files
# are each zeroed on their own most-seaward domain, so differencing two years
# subtracts a constant and flips the sign of the mean. See the markdown above.
RAW_OFFSET_DIR = HATTERAS_DATA_BASE / "2-brie-offset" / "raw_offsets"



### 9.5 Report -- and the annotation guard
The guard is the point: a swapped-in empty `AnnotationConfig` would strip
every figure's geography silently, so it raises here where the cause is one
line away. Then the resolved config and whether the baseline was found.

In [ ]:
# =============================================================================
# Report
# =============================================================================
# A swapped-in empty AnnotationConfig would strip every figure's geography
# silently; fail here instead, where the cause is one line away.
_ann_populated = any([HATTERAS_ANNOTATIONS.town_spans,
                      HATTERAS_ANNOTATIONS.village_lines,
                      HATTERAS_ANNOTATIONS.piers,
                      HATTERAS_ANNOTATIONS.groins,
                      HATTERAS_ANNOTATIONS.shoal_zones])
if not _ann_populated:
    raise RuntimeError(
        "HATTERAS_ANNOTATIONS is empty -- every figure would render with no "
        "geographic layer and no error. Check the import in section 1.")

reports.figure_config_report(
    annotations=HATTERAS_ANNOTATIONS, loess_config=LOESS_CONFIG,
    gif_config=gif_config, gif_jobs=GIF_JOBS,
    flip_sign_model=FLIP_SIGN_MODEL,
    real_domains_only=PLOT_REAL_DOMAINS_ONLY,
    groin_enabled=GROIN_ENABLED, baseline_npy=GIF_BASELINE_NPY,
    baseline_name=GIF_BASELINE_NAME, output_base_dir=OUTPUT_BASE_DIR)

## 10. `build_cascade` and `run_cascade_simulation`

Two functions. `build_cascade` constructs the model and attaches the groin;
`run_cascade_simulation` steps it through the period, applying historical
management, and writes the run's artifacts. Both take every input explicitly
rather than closing over notebook state, so the sweep scripts can call them
too.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#10-define-runcascadesimulation)

In [ ]:
# build_cascade and run_cascade_simulation both live in
# cascade_pipeline/hindcast.py, imported in section 1. The notebook,
# HAT_hindcast_1984_2024.py and HAT_groin_sweep_worker.py all build from that
# one definition, so they cannot disagree about how a model is constructed.
#
# The split is what lets section 11 hold a built-but-unstepped Cascade: BRIE's
# diffusivity and the groin's fillet prediction are only meaningful as initial
# conditions, and a prediction printed after the run is not one.
#
# `run_years` is TRANSITIONS, not states. See the module comment there for the
# off-by-one this replaced, which ran 19 updates for a 20-year period while
# dividing by 20 -- every rate came out low by 19/20.

print("build_cascade + run_cascade_simulation defined")
print(f"  run_years -> {RUN_YEARS} transitions, "
      f"time_step_count={RUN_YEARS + 1}, {RUN_YEARS + 1} annual states "
      f"({START_YEAR}-{END_YEAR})")
print("  nourishment via BN_SCHEDULE.apply_to_cascade -> "
      "cascade.nourishment_volume")
print("  road events via cascade_pipeline.roadway.apply_historical_event")

## 11. Initialize Cascade -- single config, no sweep

Section 10 defined the procedure; this section supplies the values sections
2-9 do not already produce and builds the model. Section 12 steps it.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#11-initialize-cascade----single-config-no-sweep)

### 11.1 Parameters sections 2-9 do not produce
Dune growth bounds, rebuild thresholds, wave climate, datums, and run
identity. One configuration only -- a sweep over `Hs` is a separate script.


In [ ]:
# =============================================================================
# Parameters sections 2-9 do not produce
# =============================================================================
NUM_CORES = 1        # >1 has crashed on this configuration; leave at 1

# --- dune growth (Barrier3D logistic growth bounds) --------------------------
RMIN = [0.55] * HATTERAS_DOMAINS.total_domains
RMAX = [0.95] * HATTERAS_DOMAINS.total_domains

# --- dune rebuild thresholds, m MHW ------------------------------------------
# Both are floored by roadway_manager on the first step (see the markdown).
# Stated in the documented unit rather than the original's 0.01 "# dam".
DUNE_DESIGN_ELEVATION_M = 3.0    # rebuild target
DUNE_MINIMUM_ELEVATION_M = 0.0   # rebuild trigger: let CASCADE's berm floor
                                 # govern, explicitly rather than by accident
DUNE_DESIGN_ELEVATION = [DUNE_DESIGN_ELEVATION_M] * HATTERAS_DOMAINS.total_domains
DUNE_MINIMUM_ELEVATION = [DUNE_MINIMUM_ELEVATION_M] * HATTERAS_DOMAINS.total_domains

# --- roadway ------------------------------------------------------------------
ROAD_ELEVATION = road_elevation_full   # per-domain, m MHW, from section 5
ROAD_WIDTH = 20.0

# --- wave climate: one Hs, no sweep ------------------------------------------
# A sweep over Hs is a separate script; this notebook runs one configuration.
# From hat_run.yaml (physics.wave_height_Hs); reaches run_index.csv as Hs_m,
# so a changed value is recoverable from the index and not only from the file.
Hs = RUN_CONFIG.hs                    # m, calibration value 2.5
FIXED_WAVE_PERIOD = 8                 # s
FIXED_WAVE_ASYMMETRY = 0.7
FIXED_WAVE_ANGLE_HIGH_FRACTION = 0.1

# --- datums -------------------------------------------------------------------
BERM_ELEVATION = 1.7    # m NAVD88, Hatteras Island, NCDOT-derived via NC State
MHW_ELEVATION = 0.36    # m NAVD88, Duck NC gauge (NOAA 8651370)

# --- sandbags: off for the hindcast ------------------------------------------
# From hat_run.yaml (top-level `sandbags`: a management decision, not a
# property of the coast); reaches run_index.csv as sandbags_on.
ENABLE_SANDBAG_PLACEMENT = RUN_CONFIG.sandbags
SANDBAG_MANAGEMENT_ON = [ENABLE_SANDBAG_PLACEMENT] * HATTERAS_DOMAINS.total_domains
SANDBAG_ELEVATION = 0

SEA_LEVEL_CONSTANT = True

# --- run identity, from section 9's derived name ------------------------------
# OVERWRITE is checked here, before the model is built, so a name collision
# costs nothing either way.
#
#   False  the matrix default. A directory that already holds a result
#          stops the run. 12.3 resolves the groin baseline by directory
#          name, so replacing one run's output silently redefines the
#          paired run's answer -- which is why this refuses rather
#          than asks.
#   True   iterating on one scenario: tweak a value, re-run, read the
#          figures, tweak again. The directory is EMPTIED and reused,
#          so it never mixes two trials' files, and run_index.csv's
#          row for this name is replaced. The previous trial is gone.
#          Set it back to False before running the matrix.
OVERWRITE = RUN_CONFIG.overwrite

RUN_NAME = RUN_NAME_BASE
RUN_DIR = str(OUTPUT_BASE_DIR / RUN_NAME)
# Read before the guard runs: with OVERWRITE=True the guard empties the
# directory, and a silent wipe is how a trial gets mistaken for the run
# it replaced.
_replacing = run_dir_contents(RUN_DIR)
guard_run_dir(RUN_DIR, overwrite=OVERWRITE)
print(f"\nRUN_DIR               {RUN_DIR}"
      + ("   (OVERWRITE=True)" if OVERWRITE else ""))
if _replacing:
    print(f"  replaced            {len(_replacing)} file(s) from the previous run")

### 11.2 Build
The single `build_cascade` call. Every argument comes from a section above;
none is defined here.


In [ ]:
# =============================================================================
# Build
# =============================================================================
cascade = build_cascade(
    run_years=RUN_YEARS,
    name=RUN_NAME,
    storm_file=str(STORM_FILE),
    alongshore_section_count=HATTERAS_DOMAINS.total_domains,
    num_cores=NUM_CORES,
    rmin=RMIN, rmax=RMAX,
    elevation_file=ELEVATION_FILE_PATHS,
    dune_file=DUNE_FILE_PATHS,
    dune_design_elevation=DUNE_DESIGN_ELEVATION,
    dune_minimum_elevation=DUNE_MINIMUM_ELEVATION,
    road_ele=ROAD_ELEVATION,
    road_width=ROAD_WIDTH,
    road_setback=road_setbacks_full,
    overwash_filter=OVERWASH_FILTER,
    overwash_to_dune=OVERWASH_TO_DUNE,
    nourishment_volume=NOURISHMENT_VOLUME_INIT,
    background_erosion=BACKGROUND_EROSION_RATES,
    roadway_management_on=ROADWAY_MANAGEMENT_ON,
    beach_dune_manager_on=BEACH_DUNE_MANAGEMENT_ON,
    sea_level_rise_rate=SEA_LEVEL_RISE_RATE,
    sea_level_constant=SEA_LEVEL_CONSTANT,
    sandbag_management_on=SANDBAG_MANAGEMENT_ON,
    sandbag_elevation=SANDBAG_ELEVATION,
    enable_shoreline_offset=True,
    shoreline_offset=island_offset,
    wave_height=Hs,
    wave_period=FIXED_WAVE_PERIOD,
    wave_asymmetry=FIXED_WAVE_ASYMMETRY,
    wave_angle_high_fraction=FIXED_WAVE_ANGLE_HIGH_FRACTION,
    berm_elevation=BERM_ELEVATION,
    MHW=MHW_ELEVATION,
    data_base=HATTERAS_DATA_BASE,
    parameter_file=PARAMETER_FILE,
    groin_callback=GROIN_CALLBACK,
)

### 11.3 Pre-run diagnostics -- and the groin prediction
Reads the constructed model rather than the yaml, which resolves the active
profile height section 7 had to leave open. The predicted fillet amplitude and
extent are written down **before** the run; section 12 checks the emergent
extent against them. Amplitude was tuned, extent was not.


In [ ]:
# =============================================================================
# Pre-run diagnostics
# =============================================================================



R_IPL = brie_r_ipl(cascade)
_brie = cascade._brie_coupler._brie
_d_sf_m = float(_brie.d_sf)
_h_b_m = float(cascade.barrier3d[0].h_b_TS[0]) * DAM_TO_M
_profile_height_m = _d_sf_m + _h_b_m
_berm_floor_m = float(cascade.barrier3d[0].BermEl) * DAM_TO_M


# --- groin: the prediction, made before the run ------------------------------
if GROIN_CALLBACK is not None:
    (GROIN_PREDICTED_AMPLITUDE_M,
     GROIN_PREDICTED_EXTENT_DOMAINS,
     GROIN_PREDICTED_EXTENT_M) = predict_fillet(
        trapping_rate_m_yr=GROIN_CALLBACK.M,
        r_ipl=R_IPL,
        run_years=RUN_YEARS,
        dy_m=HATTERAS_DOMAINS.domain_spacing_m,
    )
else:
    GROIN_PREDICTED_AMPLITUDE_M = None
    GROIN_PREDICTED_EXTENT_DOMAINS = None
    GROIN_PREDICTED_EXTENT_M = None

reports.pre_run_report(
    run_name=RUN_NAME, run_dir=RUN_DIR, run_years=RUN_YEARS,
    start_year=START_YEAR, end_year=END_YEAR, geometry=HATTERAS_DOMAINS,
    roadway_on=ROADWAY_MANAGEMENT_ON, beach_dune_on=BEACH_DUNE_MANAGEMENT_ON,
    wave_height=Hs, d_sf_m=_d_sf_m, h_b_m=_h_b_m,
    profile_height_m=_profile_height_m, berm_floor_m=_berm_floor_m,
    dune_design_elevation_m=DUNE_DESIGN_ELEVATION_M,
    dune_minimum_elevation_m=DUNE_MINIMUM_ELEVATION_M,
    groin_callback=GROIN_CALLBACK, groin_enabled=GROIN_ENABLED, r_ipl=R_IPL,
    predicted_amplitude_m=GROIN_PREDICTED_AMPLITUDE_M,
    predicted_extent_domains=GROIN_PREDICTED_EXTENT_DOMAINS,
    predicted_extent_m=GROIN_PREDICTED_EXTENT_M,
    reach_transport_loss_m3_yr=REACH_TRANSPORT_LOSS_M3_YR)

## 12. Run the loop, verify, then figures

Steps the model section 11 built, checks what it actually did, and only then
draws anything.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#12-run-the-loop-verify-then-figures)

### 12.1 Run
The only expensive cell in the notebook. 12.2-12.5 read the finished model, so
a change to a check or a figure re-runs only those.

The guard is deliberate: Barrier3D seeds `x_s_TS` with one entry and appends
one per update, so stepping an already-stepped model would run past TMAX and
raise from deep inside Barrier3D. Re-run section 11 for a fresh model.


In [ ]:
import datetime
import time

from tqdm.auto import tqdm

from hatteras_site_config import HATTERAS_RELOCATION_CHECK_2004

GROIN_EXTENT_THRESHOLD_FRAC = 0.10   # fraction of peak effect defining "extent"

# =============================================================================
# Run
# =============================================================================
# Barrier3D seeds x_s_TS with one entry and appends one per update, so a model
# that has already been stepped has more than one. Stepping it again would run
# past TMAX and raise from deep inside Barrier3D.
if len(cascade.barrier3d[0].x_s_TS) > 1:
    raise RuntimeError(
        f"this cascade has already been stepped "
        f"({len(cascade.barrier3d[0].x_s_TS)} states). Re-run section 11 to "
        f"build a fresh model before running section 12 again.")

_t0 = time.perf_counter()
with tqdm(total=RUN_YEARS, desc=f"{RUN_NAME}", unit="yr") as _bar:
    cascade = run_cascade_simulation(
        cascade=cascade,
        run_years=RUN_YEARS,
        name=RUN_NAME,
        run_dir=RUN_DIR,
        start_year=START_YEAR,
        geometry=HATTERAS_DOMAINS,
        alongshore_section_count=HATTERAS_DOMAINS.total_domains,
        historical_road_events=HATTERAS_ROAD_EVENTS,
        relocations_enabled=ENABLE_HISTORICAL_ROAD_RELOCATIONS,
        setback_check=HATTERAS_RELOCATION_CHECK_2004,
        nourishment_schedule=BN_SCHEDULE_APPLIED,
        groin_callback=GROIN_CALLBACK,
        progress=_bar,
        save_model_state=RUN_CONFIG.save_model_state,
    )
RUN_SECONDS = time.perf_counter() - _t0
print(f"\nruntime               {RUN_SECONDS / 60:.1f} min "
      f"({RUN_SECONDS / RUN_YEARS:.1f} s per model year)")

### 12.2 Verify
Checks what the model actually did before anything is drawn: run length
against the `states - 1` denominator that produced a 5% bias in earlier runs,
nourishment read from each manager's own `_nourishment_volume_TS`, the frozen
setbacks section 6 predicted, and which roads survived.


In [ ]:
# =============================================================================
# Verify
# =============================================================================
shoreline_m = build_shoreline_matrix(cascade)
_states, _ = shoreline_m.shape

reports.run_length_report(states=_states, run_years=RUN_YEARS)

# The denominator that produced a 5% bias in every earlier run. Checked, not
# trusted -- see the section 10 markdown.
assert _states - 1 == RUN_YEARS, (
    f"span mismatch: {_states} states implies {_states - 1} years elapsed, "
    f"but RUN_YEARS is {RUN_YEARS}")
# TWO ESTIMATORS OF THE SAME THING, and they are not interchangeable.
#
#   change_rate  (x[-1] - x[0]) / RUN_YEARS. A net displacement over a span.
#                Reads two of the RUN_YEARS+1 states, so a single-year
#                excursion still present in the final state arrives at full
#                amplitude. Kept because it is the only column that exactly
#                conserves the period's net shoreline movement, and because
#                every run before 2026-08-22 was scored on it.
#   model_lrr    OLS slope through every state. This is what section 8's
#                target IS -- CoastSat's transect_lrr_full.csv holds a
#                per-transect OLS slope -- so this is the column the figures
#                and the skill metrics use.
#
# The distinction is not cosmetic here. A nourishment fill enters as an
# instantaneous step in x_s, and BRIE's Crank-Nicolson alongshore solve
# answers a step with a grid-scale mode that alternates sign along the coast
# and decays only ~39%/yr (see compute_lrr). The 2022 Avon and Buxton fills
# are two years from the end of the 2004-2024 run, so an endpoint rate
# reports that ringing as a +-1.7 m/yr sawtooth through GIS 6-15 and 22-28.
# The LRR reports about a quarter of it, and the residue is the real fill
# edge rather than the solver.
change_rate = compute_change_rate(
    shoreline_m, span_years=RUN_YEARS, flip_sign=FLIP_SIGN_MODEL)
model_lrr, model_lrr_r2 = compute_lrr(
    shoreline_m, span_years=RUN_YEARS, flip_sign=FLIP_SIGN_MODEL)

# One name for whichever estimator section 12.5 draws, resolved once here so
# no figure call picks its own.
PLOTTED_RATE = model_lrr if RATE_ESTIMATOR == "lrr" else change_rate


# --- nourishment: the model's own record, not the schedule's intent ---------
BN_REPORT = nourishment.verify_nourishment(
    cascade, BN_SCHEDULE_APPLIED, BEACH_DUNE_MANAGEMENT_ON)
reports.nourishment_report(report=BN_REPORT, run_years=RUN_YEARS)
assert BN_REPORT["ok"], "nourishment did not reach the model as scheduled"

# --- the double-management consequence section 6 predicted ------------------
reports.frozen_setbacks_report(
    double_managed=DOUBLE_MANAGED_GIS,
    rows=(nourishment.verify_setbacks_frozen(
              cascade, DOUBLE_MANAGED_GIS, HATTERAS_DOMAINS)
          if DOUBLE_MANAGED_GIS else ()))

# --- which road_offset survived ---------------------------------------------
ROAD_SUMMARY = roadway.summarise_road_management(
    cascade, HATTERAS_DOMAINS, HATTERAS_FIRST_ROAD_DOMAIN,
    HATTERAS_LAST_ROAD_DOMAIN)
_drowned, _blocked = reports.roadway_outcome_report(summary=ROAD_SUMMARY)

### 12.3 Groin: the pre-registered extent check
The emergent alongshore extent against a paired no-groin baseline, compared to
the prediction made in 11.3. This is the part that could have failed --
amplitude was tuned, extent was not. Skipped, with a reason, when the groin is
off or no baseline run exists.


In [ ]:
# =============================================================================
# Groin: the pre-registered extent check
# =============================================================================



GROIN_EXTENT = None
if GROIN_CALLBACK is None:
    print(f"\nGROIN EXTENT          skipped: groin not enabled in this run")
elif not GIF_BASELINE_NPY:
    print(f"\nGROIN EXTENT          skipped: no paired no-groin baseline")
    print(f"  expected            {GIF_BASELINE_NAME}")
    print(f"  Run once with GROIN_ENABLED = False to create it, then re-run "
          f"this cell.")
else:
    _baseline_m = np.load(GIF_BASELINE_NPY)
    GROIN_EXTENT = measure_groin_extent(
        shoreline_m, _baseline_m, HATTERAS_DOMAINS,
        GROIN_UPDRIFT_GIS, GROIN_DOWNDRIFT_GIS, GROIN_EXTENT_THRESHOLD_FRAC)
    reports.groin_extent_report(
        extent=GROIN_EXTENT, threshold_frac=GROIN_EXTENT_THRESHOLD_FRAC,
        baseline_name=GIF_BASELINE_NAME,
        predicted_extent_domains=GROIN_PREDICTED_EXTENT_DOMAINS,
        predicted_extent_m=GROIN_PREDICTED_EXTENT_M)

### 12.4 Write

The change-rate CSV, the road-management summary, the run metadata, and one
row appended to the cross-run index.

> Why this is set up the way it is: [HAT_hindcast_methods.md](HAT_hindcast_methods.md#124-write)

In [ ]:
# =============================================================================
# Write
# =============================================================================
_shoreface_depth_m = float(cascade._brie_coupler._brie.d_sf)

run = RunInfo(
    run_name=RUN_NAME, run_dir=RUN_DIR,
    start_year=START_YEAR, end_year=END_YEAR, Hs=Hs,
    flip_sign_model=FLIP_SIGN_MODEL,
    background_erosion_on=USE_BACKGROUND_EROSION,
)

_rate_csv = os.path.join(RUN_DIR, f"{RUN_NAME}_shoreline_change_rate.csv")
# Both estimators ship, so a consumer states which one it wants rather than
# inheriting whichever the pipeline happened to write. lrr_r2 rides along
# because a slope through a domain that stepped rather than trended is a
# summary worth flagging at the point of use -- the nourished domains come
# out near 0.00.
pd.DataFrame({
    "gis_domain": np.arange(HATTERAS_DOMAINS.first_gis_id,
                            HATTERAS_DOMAINS.last_gis_id + 1),
    "change_rate_m_yr": change_rate[_real],
    "lrr_m_yr": model_lrr[_real],
    "lrr_r2": model_lrr_r2[_real],
}).to_csv(_rate_csv, index=False)
print(f"\nwrote                 {os.path.basename(_rate_csv)}")

if ROAD_SUMMARY:
    _road_csv = os.path.join(RUN_DIR, "road_management_summary.csv")
    pd.DataFrame(ROAD_SUMMARY).to_csv(_road_csv, index=False)
    print(f"                      {os.path.basename(_road_csv)}")

# --- skill against the section 8 target --------------------------------------
# Both spans are reported. The end domains carry the locked source/sink values
# (tens of m/yr), so an island-wide RMSE for a calibBE or edgeBE run is
# dominated by two domains that were pinned rather than predicted -- and a
# zeroBE run has no such term. Ranking the presets on the island-wide number
# alone would mostly rank their boundary treatment.
# SKILL is the LRR one: the target is an LRR, so this is the only pairing
# that compares like with like. SKILL_ENDPOINT is kept beside it because
# calibBE and the groin M were fit before this distinction was drawn, and a
# preset's provenance is unreadable once the metric it was fit on stops being
# recorded. Expect SKILL to be slightly WORSE than SKILL_ENDPOINT on most
# runs -- modelled erosion decelerates across a period, so an all-years slope
# is more erosive than an endpoint difference. That is the estimator changing,
# not the model.
SKILL = skill_vs_target(model_lrr, COASTSAT_TARGET, HATTERAS_DOMAINS)
SKILL_ENDPOINT = skill_vs_target(change_rate, COASTSAT_TARGET,
                                 HATTERAS_DOMAINS)
print(f"\nSKILL vs CoastSat     model LRR - target LRR, m/yr")
print(f"  island-wide         bias {SKILL['mean_bias_m_yr']:+.3f}   "
      f"RMSE {SKILL['rmse_m_yr']:.3f}   (n={SKILL['n_domains']})")
print(f"  interior (GIS 2-89) bias {SKILL['mean_bias_interior_m_yr']:+.3f}   "
      f"RMSE {SKILL['rmse_interior_m_yr']:.3f}   "
      f"(n={SKILL['n_domains_interior']})")
print(f"  endpoint estimator  bias "
      f"{SKILL_ENDPOINT['mean_bias_interior_m_yr']:+.3f}   "
      f"RMSE {SKILL_ENDPOINT['rmse_interior_m_yr']:.3f}   "
      f"(interior; the pre-LRR metric)")

# Where a straight line is a poor summary of the modelled trajectory, so the
# LRR is read with that in mind rather than silently. A nourished domain sits
# flat for most of the period and then steps, which no slope describes well.
# r2 is a variance ratio, so a domain that barely moved lands here too -- the
# two cases are told apart by whether the domain took a fill.
_lrr_r2_real = model_lrr_r2[_real]
_poor_fit = [int(_gis) for _gis, _r2 in zip(
    range(HATTERAS_DOMAINS.first_gis_id, HATTERAS_DOMAINS.last_gis_id + 1),
    _lrr_r2_real) if np.isfinite(_r2) and _r2 < LRR_R2_FLOOR]
print(f"\nLRR FIT QUALITY       median r2 "
      f"{np.nanmedian(_lrr_r2_real):.3f}   "
      f"{len(_poor_fit)} of {_lrr_r2_real.size} domains below "
      f"{LRR_R2_FLOOR:.2f}")
if _poor_fit:
    print(f"  a step, or no trend GIS "
          f"{', '.join(str(_g) for _g in _poor_fit[:12])}"
          f"{' ...' if len(_poor_fit) > 12 else ''}")

# --- run metadata: the scenario, plus what distinguishes this run -----------
# One structure renders both files: the .txt to read, the .json to parse.
# Built together so they cannot disagree -- the earlier version emitted only
# prose, so anything downstream had to re-parse it.
_GIT = git_provenance(PROJECT_BASE_DIR)
_TIMESTAMP = timestamp()
GENERATED_BY = "HAT_hindcast_1984_2024.ipynb"

# --- what the source/sink preset actually was -------------------------
# The preset's name and its numbers are separate facts.
# HATTERAS_BE_RATES_EDGE is a slice of HATTERAS_BE_RATES_CALIBRATED at
# the end domains, so editing an end domain to test edgeBE changes
# calibBE too -- and both keep their names. Recording the values is what
# makes two trials of one preset tell themselves apart afterwards.
_BE_NONZERO = sum(1 for _rate in DOMAIN_BE_RATES.values() if _rate)
_BE_DIGEST = values_digest(DOMAIN_BE_RATES)
_BE_EDGE_RATES = {f"rate_gis{_gis}_m_yr": DOMAIN_BE_RATES.get(_gis, 0.0)
                  for _gis in HATTERAS_BE_EDGE_DOMAINS}

_META = {
    "identity": {
        "run_name": RUN_NAME,
        "timestamp": _TIMESTAMP,
        "generated_by": GENERATED_BY,
        "runtime_min": f"{RUN_SECONDS / 60:.1f}",
        # The three that drift silently across a multi-day batch: which
        # Cascade class ran, which extractor version built the init surface,
        # and which commit produced both.
        "use_sandbox_cascade": (USE_SANDBOX_CASCADE,
                                "cascade.cascade_groin, not cascade.cascade"),
        "topo_product": TOPO_PRODUCT,
        "topo_dune_version": TOPO_DUNE_VERSION,
        "parameter_file": PARAMETER_FILE,
        "git_commit": _GIT["commit"],
        "git_branch": _GIT["branch"],
        "git_dirty": (_GIT["dirty"],
                      "True: the commit alone does not reproduce this run"),
    },
    "scenario": {label: value for label, value, _token in SCENARIO_SWITCHES},
    "period": {
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "run_years": (RUN_YEARS, "transitions"),
        "annual_states": _states,
    },
    "wave climate": {
        "wave_height_m": Hs,
        "wave_period_s": FIXED_WAVE_PERIOD,
        "wave_asymmetry": FIXED_WAVE_ASYMMETRY,
        "wave_angle_high_frac": FIXED_WAVE_ANGLE_HIGH_FRACTION,
        "shoreface_depth_m": (f"{_shoreface_depth_m:.2f}", "8.9 * Hs"),
    },
    "sea level": {
        "rslr_m_yr": SEA_LEVEL_RISE_RATE,
        "rslr_constant": SEA_LEVEL_CONSTANT,
    },
    "dunes and roadway": {
        "rmin / rmax": f"{RMIN[0]} / {RMAX[0]}",
        "dune_design_ele_m_mhw": (DUNE_DESIGN_ELEVATION_M,
                                  "floored by roadway_manager"),
        "dune_min_ele_m_mhw": (DUNE_MINIMUM_ELEVATION_M,
                               "floored by roadway_manager"),
        "road_width_m": ROAD_WIDTH,
        "relocations_enabled": ENABLE_HISTORICAL_ROAD_RELOCATIONS,
    },
    "source/sink": {
        "preset": SOURCE_SINK_PRESET,
        "background_erosion_on": (USE_BACKGROUND_EROSION,
                                  "implied by the preset, checked in 4.3"),
        "domains_specified": len(DOMAIN_BE_RATES),
        "nonzero_domains": _BE_NONZERO,
        "values_digest": (_BE_DIGEST,
                          "changes if any rate changed, interior included"),
        **_BE_EDGE_RATES,
    },
    "groin": {"enabled": GROIN_ENABLED},
    "skill": {
        "target": (f"CoastSat LOESS {TARGET_WINDOW}-domain",
                   "raw means over GIS 1-"
                   f"{LOESS_CONFIG.skip_southern_domains}"),
        "estimator": (RATE_ESTIMATOR,
                      "OLS slope through every annual state, matching the "
                      "target's own definition"),
        "mean_bias_m_yr": f"{SKILL['mean_bias_m_yr']:+.4f}",
        "rmse_m_yr": f"{SKILL['rmse_m_yr']:.4f}",
        "mean_bias_interior_m_yr": (
            f"{SKILL['mean_bias_interior_m_yr']:+.4f}",
            "GIS 2-89: the locked end domains excluded"),
        "rmse_interior_m_yr": f"{SKILL['rmse_interior_m_yr']:.4f}",
        "lrr_r2_median": (f"{np.nanmedian(_lrr_r2_real):.4f}",
                          "how well a line describes the modelled trajectory"),
        "lrr_r2_below_floor": (f"{len(_poor_fit)}",
                               f"domains under r2 {LRR_R2_FLOOR:.2f}"),
        "endpoint_mean_bias_interior_m_yr": (
            f"{SKILL_ENDPOINT['mean_bias_interior_m_yr']:+.4f}",
            "the pre-LRR estimator; calibBE and groin M were fit on this"),
        "endpoint_rmse_interior_m_yr":
            f"{SKILL_ENDPOINT['rmse_interior_m_yr']:.4f}",
    },
    "verification": {
        "nourishment_ok": BN_REPORT["ok"],
        "roads_drowned": len(_drowned),
        "roads_reloc_blocked": len(_blocked),
    },
}

if GROIN_CALLBACK is not None:
    _META["groin"].update({
        "trapping_rate_m_yr": GROIN_CALLBACK.M,
        "updrift / downdrift": f"GIS {GROIN_UPDRIFT_GIS} / {GROIN_DOWNDRIFT_GIS}",
        "install_year": GROIN_CALLBACK.install_year,
        "deterioration": f"{GROIN_CALLBACK.deterioration_mode}, "
                         f"floor {GROIN_CALLBACK.deterioration_fraction}",
        "r_ipl_t0": f"{R_IPL:.4f}",
        "predicted_extent_m": f"{GROIN_PREDICTED_EXTENT_M:.0f}",
    })
    if GROIN_EXTENT is not None:
        _META["groin"]["measured_extent_m"] = (
            f"{GROIN_EXTENT['updrift_m']:.0f} updrift / "
            f"{GROIN_EXTENT['downdrift_m']:.0f} downdrift")

_meta_txt, _meta_json = write_run_metadata(
    RUN_DIR, RUN_NAME, _META,
    header=[f"CASCADE run metadata -- generated by {GENERATED_BY}, "
            f"section 12.",
            "Companion .json holds the same values, machine-readable."])
print(f"                      {_meta_txt.name}")
print(f"                      {_meta_json.name}")

# --- cross-run index: one row per run, for comparing the matrix --------------
# Keyed on run_name, so a re-run replaces its row rather than adding a second.
# This is what makes 12 runs comparable without opening 12 metadata files.
# At OUTPUT_ROOT rather than the period directory: run_name already
# carries the period stem, so one file covers the whole matrix and the
# two periods stay comparable in a single table.
RUN_INDEX_PATH = OUTPUT_ROOT / RUN_INDEX_FILENAME
_index_row = {
    "run_name": RUN_NAME,
    "timestamp": _TIMESTAMP,
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "source_sink_preset": SOURCE_SINK_PRESET,
    "be_nonzero_domains": _BE_NONZERO,
    "be_values_digest": _BE_DIGEST,
    **{f"be_{_key}": _value for _key, _value in _BE_EDGE_RATES.items()},
    "scenario": SCENARIO,
    "scenario_overridden": bool(_SCENARIO_DEPARTURES),
    "groin_enabled": GROIN_ENABLED,
    "groin_trapping_m_yr": (GROIN_CALLBACK.M if GROIN_CALLBACK is not None
                            else np.nan),
    "roadway_management": ENABLE_ROADWAY_MANAGEMENT,
    "relocations_enabled": ENABLE_HISTORICAL_ROAD_RELOCATIONS,
    "beach_dune_management": ENABLE_BEACH_DUNE_MANAGEMENT,
    "nourishment_fills": ENABLE_NOURISHMENT_FILLS,
    "bdm_domains": int(sum(BEACH_DUNE_MANAGEMENT_ON)),
    "nourishment_projects": len(BN_SCHEDULE_APPLIED.projects),
    "Hs_m": Hs,
    "sandbags_on": ENABLE_SANDBAG_PLACEMENT,
    "rslr_m_yr": SEA_LEVEL_RISE_RATE,
    "annual_states": _states,
    "run_complete": _states == RUN_YEARS + 1,
    "rate_estimator": RATE_ESTIMATOR,
    "mean_bias_m_yr": SKILL["mean_bias_m_yr"],
    "rmse_m_yr": SKILL["rmse_m_yr"],
    "mean_bias_interior_m_yr": SKILL["mean_bias_interior_m_yr"],
    "rmse_interior_m_yr": SKILL["rmse_interior_m_yr"],
    "lrr_r2_median": float(np.nanmedian(_lrr_r2_real)),
    "lrr_r2_below_floor": len(_poor_fit),
    "endpoint_mean_bias_interior_m_yr":
        SKILL_ENDPOINT["mean_bias_interior_m_yr"],
    "endpoint_rmse_interior_m_yr": SKILL_ENDPOINT["rmse_interior_m_yr"],
    "nourishment_ok": BN_REPORT["ok"],
    "roads_drowned": len(_drowned),
    "roads_reloc_blocked": len(_blocked),
    "groin_extent_updrift_m": (GROIN_EXTENT["updrift_m"]
                               if GROIN_EXTENT is not None else np.nan),
    "groin_extent_downdrift_m": (GROIN_EXTENT["downdrift_m"]
                                 if GROIN_EXTENT is not None else np.nan),
    "runtime_min": round(RUN_SECONDS / 60, 1),
    "use_sandbox_cascade": USE_SANDBOX_CASCADE,
    "topo_product": TOPO_PRODUCT,
        "topo_dune_version": TOPO_DUNE_VERSION,
    "git_commit": _GIT["commit"][:12],
    "git_dirty": _GIT["dirty"],
}
RUN_INDEX = append_run_index(RUN_INDEX_PATH, _index_row)
print(f"                      {RUN_INDEX_FILENAME}  "
      f"({len(RUN_INDEX)} runs indexed)")


### 12.5 Figures
Site config arrives via section 9's bundles; omitting them would silently
strip the geographic layer.


In [ ]:
# =============================================================================
# Figures
# =============================================================================
# Site config arrives via section 9's bundles; omitting them would silently
# strip the geographic layer (DEFAULT_ANNOTATIONS is empty).
plot_rate_comparison(
    PLOTTED_RATE, cs_series, run,
    real_domains_only=PLOT_REAL_DOMAINS_ONLY, estimator=RATE_ESTIMATOR,
    sea_level_rise_rate_m_yr=SEA_LEVEL_RISE_RATE,
    save_path=os.path.join(
        RUN_DIR, f"{RUN_NAME}_shoreline_change_rate"
        f"{'_REAL_DOMAINS_ONLY' if PLOT_REAL_DOMAINS_ONLY else ''}.png"),
    show=SHOW_FIGURES, **RATE_FIG_KWARGS)

plot_annotated_rate_comparison(
    PLOTTED_RATE, cs_series, run,
    estimator=RATE_ESTIMATOR,
    sea_level_rise_rate_m_yr=SEA_LEVEL_RISE_RATE,
    save_path=os.path.join(RUN_DIR, f"{RUN_NAME}_annotated.png"),
    show=SHOW_FIGURES, **RATE_FIG_KWARGS)

# Section 9.4's validation target, resolved now that the run has a year 0.
# Buffers are NaN, so the comparison below is over the real domains only.
SHORELINE_TARGET_M, OBSERVED_CHANGE_M = build_shoreline_target(
    shoreline_m[0], START_YEAR, END_YEAR, HATTERAS_DOMAINS, RAW_OFFSET_DIR)

reports.target_misfit_report(
    target_m=SHORELINE_TARGET_M, observed_change_m=OBSERVED_CHANGE_M,
    shoreline_m=shoreline_m, end_year=END_YEAR, geometry=HATTERAS_DOMAINS,
    raw_offset_dir=RAW_OFFSET_DIR)

GIF_PATHS = make_all_shoreline_gifs(
    shoreline_m, run, GIF_JOBS,
    baseline_npy=GIF_BASELINE_NPY,
    target_m=SHORELINE_TARGET_M, **GIF_KWARGS)

print(f"\ndone                  {RUN_DIR}")

### 12.6 Animations, inline
`make_all_shoreline_gifs` writes the GIFs to `RUN_DIR` and returns their paths;
this plays them here. Each embedded animation is written into the `.ipynb`
itself, so `GIF_INLINE_MAX_MB` caps how large a file gets inlined -- anything
over that is named, not embedded.

Re-runnable on its own: with no GIFs made this session it falls back to
whatever `RUN_DIR` already holds, so a finished run can be reviewed without
stepping the model again.


In [ ]:
# =============================================================================
# Animations, inline
# =============================================================================
# A still PNG can be resampled to a display width before embedding. An
# animated GIF cannot, without re-encoding every frame -- so the file's own
# bytes are embedded and the width is set on the <img> tag instead.
GIF_DISPLAY_WIDTH = 900   # rendered width; source frames are ~1000-2000 px
GIF_INLINE_MAX_MB = 4.0   # per file -- embedded animations live in the .ipynb


def show_gif(path, width=GIF_DISPLAY_WIDTH, max_mb=GIF_INLINE_MAX_MB):
    """Plays one saved GIF inline, embedded as a data URI.

    Args:
        path: Path to the .gif.
        width: Display width in pixels.
        max_mb: Files larger than this are reported but not embedded, so one
            long high-stride animation cannot bloat the notebook.

    Returns:
        True if the animation was embedded, False if it was skipped.
    """
    path = Path(path)
    size_mb = path.stat().st_size / 1e6
    print(f"{path.name}  ({size_mb:.1f} MB)")
    if size_mb > max_mb:
        print(f"  over GIF_INLINE_MAX_MB={max_mb} -- not embedded; open {path}")
        return False
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/gif;base64,{data}" width="{width}">'))
    return True


# Fall back to what is already on disk, so this cell alone re-displays a
# finished run without re-running section 12.1.
_gifs = [Path(p) for p in GIF_PATHS] or sorted(Path(RUN_DIR).glob("*.gif"))
if not _gifs:
    print(f"no GIFs in {RUN_DIR} -- check the [GIF] messages from section 12.5")
for _gif in _gifs:
    show_gif(_gif)